<!-- notebook-header -->
<a id="top"></a>

# 10. Matrices, Math & Bits

*LeetCode Playbook · notebook 10 of 12* · Index arithmetic on grids, bit arrays and bit tricks, and the math recipes interviews use.

**In this notebook:** [Matrices](#s21) · [Math, Bits & Geometry](#s22)

← [Graphs](09_Graphs.ipynb) · [All notebooks](01_Start_Here.ipynb#notebooks) · [A-Z problem finder](12_Interview_Day.ipynb#finder) · [Design Problems](11_Design.ipynb) →

Run the setup cell below first; after that every section runs from its first cell.

In [ ]:
# Setup: everything this notebook imports. Run this cell first.
from collections import Counter, defaultdict, deque, OrderedDict
from functools import cmp_to_key, lru_cache
from itertools import accumulate, combinations, permutations, product
import bisect, heapq, math, random, string

<a id="s21"></a>

## Matrices

> A grid is a list of rows, and every matrix trick is index arithmetic on `(r, c)`: the row first, counting down; the column second, counting right.

**Reach for it when** the input is a grid, board or image and the problem says *in place* or *O(1) extra space*, *rotate*, *spiral* or *clockwise*, *diagonal*, *set its entire row and column*, *every row and every column is sorted*, or *all cells update at the same time*. Searching a grid for regions or shortest paths is BFS and DFS, in [Graphs I](09_Graphs.ipynb#s17); this section is about moving indices around.

**In this repo:** `math_geometry/` (Rotate Image, Spiral Matrix, Set Matrix Zeroes) · basics: `practice/simple/basics/matrices/01_rotate_image.py`, `02_spiral_matrix.py`, `03_set_matrix_zeroes.py`, `04_search_2d_matrix_staircase.py`, `05_game_of_life_in_place.py`, `06_matrix_traversal_patterns.py` · grid walks with bounds checks in the bank: `practice/simple/39_word_search.py`, `practice/simple/41_number_of_islands.py`, `practice/simple/44_rotting_oranges.py` · diagonals as `r - c` and `r + c`: `practice/simple/40_n_queens.py`

### The picture

Every tool in this section is arithmetic on a pair of indices, so the picture comes first: which number is the row, which is the column, what one step is, and what the two kinds of diagonal are called.

```text
grid = [[1,  2,  3,  4],          R = len(grid)    = 3 rows
        [5,  6,  7,  8],          C = len(grid[0]) = 4 columns
        [9, 10, 11, 12]]

          c=0  c=1  c=2  c=3
   r=0  [  1    2    3    4 ]     grid[1][2] == 7: the row first (down), then the column (right)
   r=1  [  5    6   (7)   8 ]     its 4 neighbours: up (0, 2), down (2, 2), left (1, 1), right (1, 3)
   r=2  [  9   10   11   12 ]     a step is (r + dr, c + dc); it is inside iff 0 <= r < R and 0 <= c < C

   "\" diagonals: r - c is constant          "/" anti-diagonals: r + c is constant
          0  -1  -2  -3                             0   1   2   3
          1   0  -1  -2                             1   2   3   4
          2   1   0  -1                             2   3   4   5
```

The main walk of this section, the spiral, peels the grid like an onion: the outer ring first, then the ring inside it.

```text
            left                  right
             v                      v
   top  ->   1 -->  2 -->  3 -->    4       lap 1: top row ->, right column down, bottom row <-,
                                    |              left column up. Each wall steps inward right
             5 -->  6 -->  7        8              after its side is walked, so the next side starts
             ^                      |              one cell later and no corner is read twice.
 bottom ->   9 <-- 10 <-- 11 <--   12       lap 2: only row 1, columns 1..2 is left: 6, 7
```

Reading every cell is already O(R·C), so most matrix problems are not about time. They are about memory: the brute force keeps a second grid, and the optimal version replaces it with one observation about the grid itself. In place means exactly that: no second grid.

Rotate Image asks for a quarter-turn clockwise of an n x n grid, in place; the brute force writes a rotated copy. The observation: a quarter-turn is two mirror flips, and a flip is only swaps.

Spiral Matrix asks for every cell in clockwise spiral order; the brute force walks with a `visited` grid. The observation: the unvisited cells always form a rectangle, so four numbers describe them.

Set Matrix Zeroes asks you to zero the whole row and column of every 0, in place; the brute force keeps a copy, or two sets of flags. The observation: a row that holds a zero is doomed anyway, so its first cell can carry the flag.

Game of Life asks for the next generation of a 0/1 board, every cell computed from the old board; the brute force copies the board. The observation: a 0/1 cell has spare bits to hold tomorrow's value.

The staircase search is the one problem here that is about time. Finding a value in a matrix whose rows and columns are both sorted costs O(R·C) by scanning; from a corner, one comparison drops a whole row or column, and the search costs O(R + C).

### From idea to code

Every tool here is index arithmetic on a grid, and two habits come before any of it. A new grid gets a fresh row on every turn, `[[0] * C for _ in range(R)]`, and a copy is made row by row, `[row[:] for row in grid]`; [Python Toolkit](01_Start_Here.ipynb#s02) shows how the shorter spellings share one row between two names.

The rest of the vocabulary is the four directions and the bounds check. `neighbours` lists the cells around `(r, c)` that exist: on the 3 x 4 grid above, the inner cell `(1, 2)` has four and the corner `(0, 0)` has two. The check runs before the read, because a negative index does not crash in Python.

In [ ]:
DIRS = [(-1, 0), (1, 0), (0, -1), (0, 1)]               # up, down, left, right


def neighbours(grid, r, c):
    R, C = len(grid), len(grid[0])
    out = []
    for dr, dc in DIRS:
        nr, nc = r + dr, c + dc
        if 0 <= nr < R and 0 <= nc < C:                  # check BEFORE reading grid[nr][nc]
            out.append((nr, nc))
    return out


grid = [[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12]]
print(neighbours(grid, 0, 0), neighbours(grid, 1, 2))   # [(1, 0), (0, 1)] [(0, 2), (2, 2), (1, 1), (1, 3)]

**Try it**
- Delete the bounds check in `neighbours` and print `[grid[nr][nc] for nr, nc in neighbours(grid, 0, 0)]`: no error, you get `[9, 5, 4, 2]`. Python reads `grid[-1][0]` as the *last* row, so a missing check silently wraps around. Only the far side crashes, and only when you read the cells: `[grid[nr][nc] for nr, nc in neighbours(grid, 2, 3)]` raises `IndexError` at `(3, 3)`.
- Predict, then print `[[len(neighbours(grid, r, c)) for c in range(4)] for r in range(3)]`: `[[2, 3, 3, 2], [3, 4, 4, 3], [2, 3, 3, 2]]`. Corners have 2, edge cells 3 and inner cells 4, and the total, 34, counts each of the 17 touching pairs twice.
- Make an 8-direction list `DIRS8 = DIRS + [(-1, -1), (-1, 1), (1, -1), (1, 1)]` and loop over it instead: the corner `(0, 0)` has 3 neighbours, the inner cell `(1, 1)` has 8.

Now the walk with the most index decisions in it, the spiral. The idea in one sentence: *the cells you have not visited yet always form a rectangle, so walk its four sides in order and move each wall inward right after you walk it.*

The **State** is four walls, `top, bottom, left, right`, and the output list. Their **Definition** is exact: the unvisited cells are rows `top..bottom` by columns `left..right`, all four inclusive. The **Invariant** follows from it: the unvisited cells form a rectangle, empty exactly when `top > bottom` or `left > right`, and every cell outside it has been output once, in spiral order. **Init** is `top, bottom, left, right = 0, R - 1, 0, C - 1`, the whole grid unvisited.

A **Step** walks one side, then moves that wall in by one; the **Record** happens on the way, each cell appended before its wall moves. A side must still exist: a row needs `top <= bottom`, a column `left <= right`. The `while` test guarantees both for the top row and the right column, but `top += 1` can use up the last row and `right -= 1` the last column, so the bottom row and the left column check again first. The **Return** is the output once the walls cross, `[]` for an empty grid.

Spiral Matrix asks for every cell of an R x C grid in clockwise spiral order, starting top-left and moving right: `[[1, 2, 3], [4, 5, 6], [7, 8, 9]]` gives `[1, 2, 3, 6, 9, 8, 7, 4, 5]`. In code the four sides are four `range` calls; a `range` never includes its stop, so walking left down to `left` needs the stop `left - 1`. On every side, read while the wall still marks the side, then move the wall: moving it first skips that side, and moving it one side too late reads the corner twice.

In [ ]:
def spiral_order(grid):
    if not grid or not grid[0]:
        return []
    out = []                                         # STATE: the cells so far, in spiral order
    top, bottom = 0, len(grid) - 1                   # STATE + INIT: unvisited = rows top..bottom,
    left, right = 0, len(grid[0]) - 1                #   columns left..right (all inclusive)
    while top <= bottom and left <= right:           # the rectangle is not empty
        for c in range(left, right + 1):             # top row, left -> right
            out.append(grid[top][c])                 # RECORD as we walk past
        top += 1                                     # STEP: the top wall moves in
        for r in range(top, bottom + 1):             # right column, going down
            out.append(grid[r][right])               # RECORD
        right -= 1                                   # STEP: the right wall moves in
        if top <= bottom:                            # a row is still left
            for c in range(right, left - 1, -1):     # bottom row, right -> left
                out.append(grid[bottom][c])          # RECORD
            bottom -= 1                              # STEP: the bottom wall moves in
        if left <= right:                            # a column is still left
            for r in range(bottom, top - 1, -1):     # left column, going up
                out.append(grid[r][left])            # RECORD
            left += 1                                # STEP: the left wall moves in
    return out                                       # RETURN: the walls have crossed


print(spiral_order(grid))                            # [1, 2, 3, 4, 8, 12, 11, 10, 9, 5, 6, 7]
print(spiral_order([[1, 2, 3], [4, 5, 6], [7, 8, 9]]))   # [1, 2, 3, 6, 9, 8, 7, 4, 5]

**Try it**
- Delete the line `if top <= bottom:` (keep the loop under it, dedented) and run `spiral_order([[1, 2, 3]])`: `[1, 2, 3, 2, 1]`. With one row left, the "bottom row" is the top row you already walked, walked back.
- Do the same with `if left <= right:` and run `spiral_order([[1], [2], [3]])`: `[1, 2, 3, 2]`. The single column is walked down and then partly back up.
- Change `range(right, left - 1, -1)` to `range(right, left, -1)`: on `grid` the 9 goes missing. A `range` never includes its stop, so walking *down to* `left` needs `left - 1`.
- Print `(top, bottom, left, right)` at the top of the `while` for a 4 x 4 grid: `(0, 3, 0, 3)` then `(1, 2, 1, 2)`, one lap per ring.

### Watch it work

The two guards are the lines that go wrong, so watch them act. Each block is one lap and each line is one side of the current rectangle; on the 3 x 4 grid the second lap has a single row left, so the right column comes up empty and the bottom-row guard skips its side.

In [ ]:
def trace_spiral(grid):
    top, bottom, left, right = 0, len(grid) - 1, 0, len(grid[0]) - 1
    while top <= bottom and left <= right:
        print(f"walls: rows {top}..{bottom}, columns {left}..{right}")
        print("   top row    ->", [grid[top][c] for c in range(left, right + 1)])
        top += 1
        print("   right col  v ", [grid[r][right] for r in range(top, bottom + 1)])
        right -= 1
        if top <= bottom:
            print("   bottom row <-", [grid[bottom][c] for c in range(right, left - 1, -1)])
            bottom -= 1
        else:
            print("   bottom row    skipped: no row left")
        if left <= right:
            print("   left col   ^ ", [grid[r][left] for r in range(bottom, top - 1, -1)])
            left += 1
        else:
            print("   left col      skipped: no column left")


trace_spiral(grid)

**Try it**
- Predict, then run `trace_spiral([[1, 2], [3, 4], [5, 6], [7, 8]])`: a single lap, `1 2 | 4 6 8 | 7 | 5 3`. After it `left = 1 > right = 0`, so the walls have crossed.
- Run `trace_spiral([[5]])`: the top row takes the only cell, and both guards skip their side.
- Move `top += 1` below the right-column `print` (the wall moves one side too late): the right column becomes `[4, 8, 12]`, so the corner 4 is read twice. The same edit in `spiral_order` gives `[1, 2, 3, 4, 4, 8, 12, 11, 10, 9, 5, 6, 7, 7]`.
- Run it on a 5 x 5 grid, `[[5 * r + c for c in range(5)] for r in range(5)]`: three laps, and the last one is just the centre cell 12.

### The rest of the toolbox

The spiral taught the habit: say exactly what each index means, then move it. The other four problems use the same habit with a different index map, so the maps come first, as two tables to look things up in.

Derive any map in thirty seconds from two cells. Say where `(0, 0)` and `(0, 1)` must land, then read the formula off. Clockwise, the top-left corner goes to the top-right corner of the new grid, `(0, R-1)`, and one step right becomes one step down, so `(r, c)` goes to `(c, R-1-r)`. Always check a map on a non-square grid, where a wrong `R` or `C` shows up at once.

| Move | `(r, c)` goes to | New shape | A new grid in one line |
|---|---|---|---|
| transpose | `(c, r)` | C x R | `[list(t) for t in zip(*g)]` |
| rotate clockwise | `(c, R-1-r)` | C x R | `[list(t) for t in zip(*g[::-1])]` |
| rotate counter-clockwise | `(C-1-c, r)` | C x R | `[list(t) for t in zip(*g)][::-1]` |
| rotate 180° | `(R-1-r, C-1-c)` | R x C | `[row[::-1] for row in g[::-1]]` |
| mirror left-right | `(r, C-1-c)` | R x C | `[row[::-1] for row in g]` |

| Name for a cell | Key | All cells with one key |
|---|---|---|
| flat index | `i = r * C + c` | back: `r, c = divmod(i, C)` |
| `\` diagonal | `k = r - c` | `for r in range(max(0, k), min(R, C + k)):` with `c = r - k` |
| `/` anti-diagonal | `d = r + c` | `for r in range(max(0, d - C + 1), min(R, d + 1)):` with `c = d - r` (keeps `0 <= c < C`) |
| ring (layer) | `min(r, c, R-1-r, C-1-c)` | its distance to the nearest edge |
| 3 x 3 box (sudoku) | `(r // 3) * 3 + c // 3` | boxes numbered 0..8, row by row |

Rotation comes first because its map is the model for the others. Rotate Image asks for a quarter-turn clockwise of an n x n matrix, in place: `[[1, 2, 3], [4, 5, 6], [7, 8, 9]]` becomes `[[7, 4, 1], [8, 5, 2], [9, 6, 3]]`. The turn sends `(r, c)` to `(c, n-1-r)`, and that move is two mirror flips in a row, each of them only swaps. Mirror across the main diagonal first, the transpose, which sends `(r, c)` to `(c, r)`; then mirror left-right by reversing each row, which sends `(c, r)` to `(c, n-1-r)`.

```text
  1 2 3                      1 4 7                        7 4 1
  4 5 6   -- transpose -->   2 5 8   -- reverse rows -->  8 5 2
  7 8 9    swap across "\"   3 6 9     flip left-right    9 6 3
```

The other classic way turns four cells at once, ring by ring. Call them `a, b, d, e`, because the name `c` is taken by the column. Each one is the previous one with the same map applied: `b` is where `a` goes, `d` is where `b` goes, `e` is where `d` goes, and `e` goes back to `a`.

```text
   n = 4, ring 0, c = 1:      . a . .        a = (0, 1) -> b = (1, 3)
                              . . . b        b = (1, 3) -> d = (3, 2)
                              e . . .        d = (3, 2) -> e = (2, 0)
                              . . d .        e = (2, 0) -> a
```

In the code, every cell on the left receives the value of the cell that turns into it, so the right side is the left side shifted one place, last first: `a, b, d, e` receive `e, a, b, d`. Both functions turn the same 3 x 3 grid, and the last line checks that they agree.

In [ ]:
def rotate_clockwise(m):                       # n x n, in place
    n = len(m)
    for r in range(n):
        for c in range(r + 1, n):              # above the diagonal only: each pair swapped once
            m[r][c], m[c][r] = m[c][r], m[r][c]
    for row in m:
        row.reverse()                          # then mirror left <-> right


def rotate_by_rings(m):                        # the same turn, four cells per move
    n = len(m)
    for r in range(n // 2):                    # ring r (the centre of an odd n stays put)
        for c in range(r, n - 1 - r):          # the ring's top edge, minus its last cell
            m[r][c], m[c][n - 1 - r], m[n - 1 - r][n - 1 - c], m[n - 1 - c][r] = \
                m[n - 1 - c][r], m[r][c], m[c][n - 1 - r], m[n - 1 - r][n - 1 - c]


a = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
b = [row[:] for row in a]
rotate_clockwise(a)
rotate_by_rings(b)
print(a)         # [[7, 4, 1], [8, 5, 2], [9, 6, 3]]
print(a == b)    # True

**Try it**
- Change `range(r + 1, n)` to `range(n)` in the transpose: you get `[[3, 2, 1], [6, 5, 4], [9, 8, 7]]`, only the mirror image. Every pair was swapped twice, which undid the transpose.
- Replace `for row in m: row.reverse()` with `m.reverse()` (reverse the *order* of the rows): `[[3, 6, 9], [2, 5, 8], [1, 4, 7]]`, the counter-clockwise turn.
- Call `rotate_clockwise` four times on the same matrix and compare it with a saved copy: four quarter-turns change nothing.
- In `rotate_by_rings`, print `(r, c)` for each move on a 4 x 4: `(0, 0) (0, 1) (0, 2) (1, 1)`. Four moves of four cells cover all 16.

Next comes a problem where the trick is not moving cells but storing facts inside the grid. Set Matrix Zeroes asks: if a cell is 0, set its entire row and column to 0, in place, with O(1) extra space; `[[1, 1, 1], [1, 0, 1], [1, 1, 1]]` becomes `[[1, 0, 1], [0, 0, 0], [1, 0, 1]]`. The answer needs only R + C facts, "row r has a zero" and "column c has a zero", and the matrix's own first column and first row can hold them.

```text
   row flags live in column 0, column flags live in row 0

     1  1  1  1                  1  0  1  1      m[0][1] = 0: "zero column 1"
     1  0  1  1   -- mark -->    0  0  1  1      m[1][0] = 0: "zero row 1"
     1  1  1  1                  1  1  1  1
```

Writing `m[r][0] = 0` is safe because it happens only when row r contains a zero, and then that whole row ends up zero anyway, so nothing is lost; the same goes for `m[0][c]` and column c. The markers do overwrite two facts, whether row 0 and column 0 held a zero *themselves*. Save those in two booleans first, and fix row 0 and column 0 last.

The code reads in four passes: save the two facts, mark, apply the marks to the inner cells, then zero the marker row and column. In the cell's grid both zeros sit in row 0, one of them in the corner, so the answer zeroes row 0, column 0 and column 3, and nothing else.

In [ ]:
def set_zeroes(m):
    R, C = len(m), len(m[0])
    row0 = any(x == 0 for x in m[0])                  # STATE: the two facts the markers overwrite
    col0 = any(m[r][0] == 0 for r in range(R))
    for r in range(1, R):                             # mark: row flags in column 0,
        for c in range(1, C):                         #       column flags in row 0
            if m[r][c] == 0:
                m[r][0] = m[0][c] = 0                 # STEP: flag row r and column c
    for r in range(1, R):                             # apply the flags to the inner cells
        for c in range(1, C):
            if m[r][0] == 0 or m[0][c] == 0:
                m[r][c] = 0                           # RECORD: apply the flags
    if row0:                                          # the marker row and column go LAST
        for c in range(C):
            m[0][c] = 0                               # RECORD: row 0
    if col0:
        for r in range(R):
            m[r][0] = 0                               # RECORD: column 0


z = [[0, 1, 2, 0], [3, 4, 5, 2], [1, 3, 1, 5]]
set_zeroes(z)
print(z)    # [[0, 0, 0, 0], [0, 4, 5, 0], [0, 3, 1, 0]]

**Try it**
- Print `m` right after the marking loop for `[[1, 1, 1], [1, 0, 1], [1, 1, 1]]`: `[[1, 0, 1], [0, 0, 1], [1, 1, 1]]`. The flags sit in row 0 and column 0, exactly as in the picture.
- Start the two apply loops at 0 (`range(R)` and `range(C)`) and rerun the cell: `z` comes out all zeros instead of `[[0, 0, 0, 0], [0, 4, 5, 0], [0, 3, 1, 0]]`. Row 0 is applied first, and its new zeros then look like flags for every column.
- Try the shortcut that drops the booleans: mark from every cell (`range(R)` and `range(C)` in the marking loops), then let `if m[0][0] == 0:` zero both row 0 and column 0. `[[1, 0], [1, 1]]` gives `[[0, 0], [0, 0]]` instead of `[[0, 0], [1, 0]]`: one cell cannot say "row 0" and "column 0" separately.
- Delete `col0` and its block, then run `[[1, 1], [0, 1]]`: you get `[[1, 1], [0, 0]]` instead of `[[0, 1], [0, 0]]`. The original 0 still works as row 1's flag, but nothing ever zeroes column 0.

The next problem uses the grid's order rather than its shape. Search a 2D Matrix II asks whether a target is in a matrix where every row is sorted left to right and every column top to bottom: `[[1, 4, 7], [2, 5, 8], [3, 6, 9]]` with target 5 gives `True`. Stand on the top-right corner, the largest value in its row and the smallest in its column.

One comparison there throws away a whole column when the corner is too big, because everything below it is even bigger, or a whole row when it is too small, because everything to its left is even smaller. It is the staircase of [Two Pointers](03_Two_Pointers_Sliding_Window_Strings.ipynb#s05), where each comparison kills a row or a column of the table of sorted pairs; here the table is the matrix itself.

```text
  target = 5                    15 > 5: the column below is all bigger   -> step left
   1   4   7  11  15            11 > 5, then 7 > 5: same                  -> left, left
   2   5   8  12  19             4 < 5: the row to its left is smaller   -> step down
   3   6   9  16  22             5 == 5: found after 5 looks (never more than R + C - 1)
  10  13  14  17  24
  18  21  23  26  30
```

After each step the cells still in play are rows `r..` and columns `..c`, and `(r, c)` is again their top-right corner, so the same argument repeats. The walk is a staircase of at most R + C − 1 looks, one comparison per step, and it ends when a wall is crossed. The cell searches the matrix of the picture: 5 is there, 20 is not.

In [ ]:
def search_sorted_matrix(m, target):
    if not m or not m[0]:
        return False
    r, c = 0, len(m[0]) - 1                    # top-right corner
    while r < len(m) and c >= 0:               # still inside
        if m[r][c] == target:
            return True
        if m[r][c] > target:
            c -= 1                             # everything below is bigger: drop the column
        else:
            r += 1                             # everything to the left is smaller: drop the row
    return False


sm = [[1, 4, 7, 11, 15], [2, 5, 8, 12, 19], [3, 6, 9, 16, 22], [10, 13, 14, 17, 24], [18, 21, 23, 26, 30]]
print(search_sorted_matrix(sm, 5), search_sorted_matrix(sm, 20))   # True False

**Try it**
- Count the looks for target 20, which is absent: 9 = R + C − 1. The walk goes 15, 19, 22, 16, 17, 26, 23, 21, 18 and steps off the bottom.
- Start at the bottom-left corner instead (`r, c = len(m) - 1, 0`, move up when too big and right when too small): same answers. That corner is also the biggest of one line and the smallest of the other. The top-left is not: both of its moves go to bigger values.
- Search for 0 and for 31: the walk leaves the grid on the left side and on the bottom side, and both return `False`.

Set Matrix Zeroes hid flags in cells it was going to overwrite anyway; Game of Life hides tomorrow's value inside a spare bit of today's. It asks for the next generation of a 0/1 board, in place: a cell lives next turn when it has 3 live neighbours, or 2 and is alive now, so the blinker `[[0, 0, 0], [1, 1, 1], [0, 0, 0]]` becomes `[[0, 1, 0], [0, 1, 0], [0, 1, 0]]`. Every cell must be computed from the *old* board, so writing a new value straight away corrupts neighbours that are read later.

```text
  value   bit 1 (tomorrow)   bit 0 (today)
    0           0                 0          dead, stays dead
    1           0                 1          alive, dies
    2           1                 0          dead, is born
    3           1                 1          alive, survives
```

A cell needs one bit, but an int has plenty: keep today in bit 0, write tomorrow into bit 1, and let neighbours read `cell & 1`. A last pass shifts tomorrow into place, and the cell's blinker turns from a row into a column. For values wider than one bit, pack two numbers in base K: `cell = old + K * new`, read the old value with `% K`, and finish with `// K`.

In [ ]:
def game_of_life(board):
    R, C = len(board), len(board[0])
    for r in range(R):
        for c in range(C):
            live = 0
            for dr in (-1, 0, 1):
                for dc in (-1, 0, 1):
                    nr, nc = r + dr, c + dc
                    if (dr or dc) and 0 <= nr < R and 0 <= nc < C:   # 8 neighbours, inside
                        live += board[nr][nc] & 1          # bit 0 = today, even if bit 1 is set
            if live == 3 or (live == 2 and board[r][c] & 1):
                board[r][c] |= 2                           # tomorrow goes into bit 1
    for r in range(R):
        for c in range(C):
            board[r][c] >>= 1                              # tomorrow becomes today


blinker = [[0, 0, 0], [1, 1, 1], [0, 0, 0]]
game_of_life(blinker)
print(blinker)    # [[0, 1, 0], [0, 1, 0], [0, 1, 0]]

**Try it**
- Print the board just before the final `>>= 1` pass: `[[0, 2, 0], [1, 3, 1], [0, 2, 0]]`. You can read births (2), deaths (1) and survivors (3) straight off it.
- Replace `board[nr][nc] & 1` with `board[nr][nc]` and rerun the blinker: you get `[[0, 1, 0], [1, 0, 1], [0, 0, 0]]`. Cells already marked 2 or 3 count as two or three neighbours, so the centre "sees" 6 and dies.
- Run `game_of_life` twice on the blinker: it flips back to the horizontal line (it has period 2).
- A 2 x 2 block `[[1, 1], [1, 1]]` never changes: every cell has exactly 3 live neighbours.

Finally the idea behind every diagonal walk: name a line by what stays constant along it. Along a "\" diagonal `r - c` stays the same, along a "/" anti-diagonal `r + c` stays the same, and the ring of a cell is its distance to the nearest edge. So "walk the cells line by line" becomes "group the cells by a key", with the keys from the table above.

`group_by` does exactly that on the 3 x 4 grid, keeping row-major order inside each line, row by row and left to right: the anti-diagonal `r + c == 2` holds `[3, 6, 9]`. A transpose swaps the roles of `r` and `c`, and for a non-square grid it must build a new C x R grid, which `zip(*grid)` does in one line.

In [ ]:
def group_by(grid, key):                       # key(r, c) -> which line the cell is on
    lines = defaultdict(list)
    for r, row in enumerate(grid):
        for c, x in enumerate(row):
            lines[key(r, c)].append(x)         # row-major order inside each line
    return [lines[k] for k in sorted(lines)]


grid = [[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12]]
R, C = len(grid), len(grid[0])
print(group_by(grid, lambda r, c: r - c))      # [[4], [3, 8], [2, 7, 12], [1, 6, 11], [5, 10], [9]]
print(group_by(grid, lambda r, c: r + c))      # [[1], [2, 5], [3, 6, 9], [4, 7, 10], [8, 11], [12]]
print(group_by(grid, lambda r, c: min(r, c, R - 1 - r, C - 1 - c)))   # [[1, 2, 3, 4, 5, 8, 9, 10, 11, 12], [6, 7]]
print([list(col) for col in zip(*grid)])       # [[1, 5, 9], [2, 6, 10], [3, 7, 11], [4, 8, 12]]

**Try it**
- Reverse every *even-numbered* anti-diagonal and flatten: `[x for k, d in enumerate(group_by(grid, lambda r, c: r + c)) for x in (d[::-1] if k % 2 == 0 else d)]` gives `[1, 2, 5, 9, 6, 3, 4, 7, 10, 11, 8, 12]`, the zigzag order of Diagonal Traverse (498).
- Toeplitz check (766), "every '\' diagonal holds one value": `all(len(set(d)) == 1 for d in group_by(t, lambda r, c: r - c))` is `True` for `t = [[1, 2, 3], [4, 1, 2], [5, 4, 1]]`. Change one cell and watch it turn `False`.
- Walk anti-diagonal `d = 3` straight from the table, without grouping: `[grid[r][d - r] for r in range(max(0, d - C + 1), min(R, d + 1))]` is `[4, 7, 10]`, the same as `group_by(grid, lambda r, c: r + c)[3]`.

### Where it goes wrong

Most matrix bugs are index bugs, and these twelve are the ones that come back.

1. **Aliased rows.** `[[0] * C] * R` is R references to one row, so `g = [[0] * 2] * 2; g[0][0] = 1` gives `[[1, 0], [1, 0]]`. Build with `[[0] * C for _ in range(R)]` and copy with `[row[:] for row in grid]`, never `grid[:]`; [Python Toolkit](01_Start_Here.ipynb#s02) shows why.
2. **Negative indexes do not crash.** `grid[-1][c]` quietly reads the last row, so a missing `0 <= nr` check gives wrong answers instead of an error: the neighbours of `(0, 0)` would include `grid[-1][0] = 9`. Check bounds *before* indexing.
3. **Rows and columns mixed up.** `grid[r][c]`, row first; `R = len(grid)`, `C = len(grid[0])`. Swap `R` and `C` in `neighbours` and, on the 3 x 4 grid, `(2, 0)` gets the neighbour `(3, 0)`, which raises `IndexError` when read, while `(1, 2)` loses `(1, 3)`. A square test grid hides this: always test one non-square grid.
4. **Transposing twice.** `for c in range(n)` swaps every pair twice, so after the row reversal `[[1, 2], [3, 4]]` becomes `[[2, 1], [4, 3]]`, a mirror. Use `range(r + 1, n)`, and after the transpose reverse each *row* (clockwise), not the row order (counter-clockwise).
5. **Spiral without the two re-checks.** A last single row or column is walked twice: `[[1, 2, 3]]` gives `[1, 2, 3, 2, 1]`.
6. **Destroying markers before using them.** In Set Matrix Zeroes, record whether row 0 and column 0 have zeros *first*, and zero them *last*. Without `col0`, `[[1, 1], [0, 1]]` gives `[[1, 1], [0, 0]]`.
7. **Overwriting a value others still need.** In Game of Life, write tomorrow into a spare bit and read `cell & 1`. Without the `& 1`, the blinker gives `[[0, 1, 0], [1, 0, 1], [0, 0, 0]]`.
8. **Staircase from the wrong corner.** From the top-left of `[[1, 4], [2, 5]]` with target 2: 1 is too small, but both moves (to 4 and to 2) go to bigger values, so there is no direction. Start top-right (or bottom-left).
9. **Negative slice stops.** `grid[bottom][right:left - 1:-1]` breaks when `left == 0`: the stop becomes −1, which means "the last item", so `[1, 2, 3, 4][3:-1:-1] == []`. Use `range(right, left - 1, -1)` or `row[left:right + 1][::-1]`.
10. **Rebinding instead of mutating.** Inside `rotate(matrix)`, `matrix = [list(t) for t in zip(*matrix[::-1])]` only rebinds the local name: the caller's `[[1, 2], [3, 4]]` is unchanged. `matrix[:] = ...` mutates it, correctly, but with O(n²) extra memory.
11. **`zip` gives tuples.** `list(zip(*[[1, 2], [3, 4]]))[0][0] = 9` raises `TypeError`: convert each one with `list(t)`.
12. **`len(grid[0])` on an empty grid** raises `IndexError`: handle `[]` and `[[]]` before anything else.

### Edge cases to say out loud

Say these before coding, because each one lands on a guard or a bounds check above. Empty grid (`[]`, `[[]]`) · 1 x 1 · a single row · a single column · non-square R ≠ C (in-place rotation needs a square) · zeros inside the marker row or column · a target smaller or bigger than everything · a board with nothing alive.

In [ ]:
def after(fn, m):                              # run an in-place function on a copy, return the copy
    m = [row[:] for row in m]
    fn(m)
    return m


assert spiral_order([]) == [] and spiral_order([[]]) == []
assert spiral_order([[7]]) == [7]
assert spiral_order([[1, 2, 3]]) == [1, 2, 3]                  # one row: the guards stop the way back
assert spiral_order([[1], [2], [3]]) == [1, 2, 3]              # one column
assert spiral_order([[1, 2], [3, 4], [5, 6]]) == [1, 2, 4, 6, 5, 3]
assert after(rotate_clockwise, [[5]]) == [[5]]
assert after(rotate_clockwise, [[1, 2], [3, 4]]) == [[3, 1], [4, 2]]
assert after(set_zeroes, [[0, 1], [1, 1]]) == [[0, 0], [0, 1]]   # the zero is in the marker corner
assert after(set_zeroes, [[1], [0], [3]]) == [[0], [0], [0]]     # a single column
assert not search_sorted_matrix([[]], 1)
assert not search_sorted_matrix(sm, 0) and search_sorted_matrix(sm, 30) and search_sorted_matrix(sm, 18)
assert after(game_of_life, [[0, 0], [0, 0]]) == [[0, 0], [0, 0]]
print("edge cases pass")

**Try it**
- Add `assert after(rotate_by_rings, [[1, 2], [3, 4]]) == [[3, 1], [4, 2]]` and run: the ring version agrees on the smallest ring.
- Predict `after(game_of_life, [[1]])` before running (`[[0]]`: a lonely cell dies), then write it as an assert.
- Feed `after(rotate_clockwise, [[1, 2, 3], [4, 5, 6]])`: no error, but `[[3, 4, 1], [6, 5, 2]]` is not a rotation (a rotated 2 x 3 grid is 3 x 2). In-place rotation only makes sense for a square.

Asserts check the cases you thought of. Simple versions that are hard to get wrong check the ones you did not: a walker that turns right whenever it is blocked, two sets for the zeroes, and the one-line rotation from the map table. The walker spends O(R·C) on a `visited` set but has no guards to forget, and it is the brute force to say first in the interview. The cell runs all three against the templates on random grids of every shape, 1 x N and N x 1 included.

In [ ]:
def spiral_brute(grid):                        # the walker: turn right when blocked
    if not grid or not grid[0]:
        return []
    R, C = len(grid), len(grid[0])
    seen, out, r, c, d = set(), [], 0, 0, 0
    TURNS = [(0, 1), (1, 0), (0, -1), (-1, 0)]  # right, down, left, up: clockwise order
    for _ in range(R * C):                      # one step per cell
        out.append(grid[r][c])
        seen.add((r, c))
        nr, nc = r + TURNS[d][0], c + TURNS[d][1]
        if not (0 <= nr < R and 0 <= nc < C) or (nr, nc) in seen:
            d = (d + 1) % 4                     # blocked: turn clockwise
            nr, nc = r + TURNS[d][0], c + TURNS[d][1]
        r, c = nr, nc
    return out


def zeroes_brute(m):                           # two sets instead of markers
    rows = {r for r, row in enumerate(m) for x in row if x == 0}
    cols = {c for row in m for c, x in enumerate(row) if x == 0}
    return [[0 if r in rows or c in cols else x for c, x in enumerate(row)] for r, row in enumerate(m)]


random.seed(0)
for _ in range(300):
    R, C = random.randint(1, 5), random.randint(1, 5)        # 1 x N and N x 1 included
    m = [[random.randint(0, 3) for _ in range(C)] for _ in range(R)]
    assert spiral_order(m) == spiral_brute(m), m
    assert after(set_zeroes, m) == zeroes_brute(m), m
    sq = [row[:min(R, C)] for row in m[:min(R, C)]]          # the largest square in the corner
    assert after(rotate_clockwise, sq) == [list(t) for t in zip(*sq[::-1])], sq
print("walls, markers and in-place rotation agree with the simple versions on 300 random grids")

**Try it**
- Delete the `if top <= bottom:` guard in `spiral_order` (rerun its cell), then rerun this one: the assert stops on the one-row grid `[[3, 0, 2]]` and prints it.
- Do the same with the `if left <= right:` guard: now the one-column grid `[[1], [1], [1]]` is the first to fail. Its three cells come out four times.
- Replace `zip(*sq[::-1])` with `zip(*sq)` (the transpose, not the turn): the rotation check fails on the very first grid.

### Variations

The five tools combine into the common follow-ups. The table says which lines change and what each problem asks; the two cells below work out the Google favourites.

| Variation | What changes from the template | Problems |
|---|---|---|
| **Rotate counter-clockwise** | transpose, then reverse the row *order* (`m.reverse()`) | 48 Rotate Image, the other way round |
| **Rotate 180°** | reverse the row order and reverse every row | 1886 Determine Whether Matrix Can Be Obtained By Rotation: can quarter-turns make `mat` equal `target`; try all four |
| **Spiral fill** | the same four walls, writing `1, 2, 3, ...` instead of reading | 59 Spiral Matrix II: fill an n x n grid with 1..n² in spiral order |
| **Spiral outward from a cell** | step lengths 1, 1, 2, 2, 3, 3, ...; record only the steps that land inside | 885 Spiral Matrix III: spiral outward from a given cell and list the grid cells in the order they are reached |
| **Zigzag diagonals** | group by `r + c`, reverse every other group | 498 Diagonal Traverse: the anti-diagonals, alternating direction |
| **Toeplitz check** | every cell equals its up-left neighbour (`r - c` is constant) | 766 Toeplitz Matrix: does every "\" diagonal hold one value |
| **Transpose a non-square** | a new C x R grid: `[list(col) for col in zip(*grid)]` | 867 Transpose Matrix: flip a grid over its main diagonal, so its rows become columns |
| **Shift the grid k steps** | flat index: the cell at `i` moves to `(i + k) % (R * C)` | 1260 Shift 2D Grid: move every cell k places along the rows, the last cell wrapping round to the first |
| **Fully sorted (row-major) matrix** | one sorted array of R·C items: binary search on the flat index (see [Binary Search](05_Binary_Search_Sorting.ipynb#s09)) | 74 Search a 2D Matrix: each row starts after the previous row ends |
| **k-th smallest in a sorted matrix** | binary search on the value; a staircase counts the cells ≤ mid | 378 Kth Smallest Element in a Sorted Matrix: the k-th value when rows and columns are sorted, worked out below |
| **Sparse matrix multiplication** | multiply only the non-zero entries | 311 Sparse Matrix Multiplication: the product of two mostly-zero matrices, worked out below |
| **Tic-tac-toe on n x n** | one ±1 counter per row, column and diagonal, and a counter reaching ±n wins: [Design Problems](11_Design.ipynb#s24) | 348 Design Tic-Tac-Toe: report the winner after each move |
| **Queens and diagonals** | `r - c` and `r + c` as set keys, inside the search of [Backtracking](08_Backtracking.ipynb#s16) | 51, 52 N-Queens: place n queens so that none attacks another |
| **Candy Crush** | mark the crushed cells by negating them, then compact each column bottom-up with a write pointer | 723 Candy Crush: crush every run of three or more equal candies, let the rest fall, repeat |
| **Rotating the Box** | slide the stones to the right end of each row, stopping at obstacles, then rotate clockwise | 1861 Rotating the Box: where the stones settle after the box is turned clockwise |
| **Game of Life on an unbounded board** | a set of live cells; count neighbours with a `Counter` | 289 Game of Life on a board without edges, worked out below |

Kth Smallest Element in a Sorted Matrix (378) asks for the k-th smallest value of a matrix sorted by rows and by columns; in `[[1, 5, 9], [10, 11, 13], [12, 13, 15]]` the 8th smallest is 13. The cells ≤ v can be *counted* with a staircase in O(R + C): from the bottom-left, a cell ≤ v counts together with everything above it.

That count only grows with v, so binary-search the smallest v whose count reaches k, a binary search on the answer as in [Binary Search](05_Binary_Search_Sorting.ipynb#s09). That v is always a matrix value: if it were not, `count(v - 1)` would be the same, and v would not be the smallest. In the example 5 cells are ≤ 11, 6 are ≤ 12 and 8 are ≤ 13, so the 8th smallest is 13.

In [ ]:
def count_at_most(m, v):                       # staircase from the bottom-left: cells <= v
    r, c, count = len(m) - 1, 0, 0
    while r >= 0 and c < len(m[0]):
        if m[r][c] <= v:
            count += r + 1                     # this cell and everything above it
            c += 1
        else:
            r -= 1                             # too big, and so is the rest of this row
    return count


def kth_smallest(m, k):
    lo, hi = m[0][0], m[-1][-1]                # the answer is a value in [lo, hi]
    while lo < hi:
        mid = (lo + hi) // 2
        if count_at_most(m, mid) >= k:
            hi = mid                           # mid is enough: the answer is <= mid
        else:
            lo = mid + 1                       # too few cells <= mid: the answer is bigger
    return lo


sorted_m = [[1, 5, 9], [10, 11, 13], [12, 13, 15]]
print(kth_smallest(sorted_m, 8), [count_at_most(sorted_m, v) for v in (11, 12, 13)])   # 13 [5, 6, 8]

**Try it**
- `count_at_most(sorted_m, 14)` is also 8, but 14 is not in the matrix. It can never be the answer: 13 already reaches the same count, and the search looks for the *smallest* such v.
- Change `hi = mid` to `hi = mid - 1`: `kth_smallest(sorted_m, 6)` returns 11 instead of 12. `mid = 12` was itself the answer, and it got thrown away.
- `kth_smallest(sorted_m, 7)` and `kth_smallest(sorted_m, 8)` are both 13: duplicates take one rank each.

Two Google favourites remain. Sparse Matrix Multiplication (311) multiplies two matrices that are mostly zeros; a zero adds nothing to `out[i][j] = sum(A[i][k] * B[k][j])`, so keep only the non-zero entries of each row of B and skip the zeros of A. Game of Life on an unbounded board drops the grid: keep the *set* of live cells and let each one vote for its 8 neighbours, so that a `Counter` of the votes is the neighbour count, and the blinker's horizontal three cells become the vertical three.

In [ ]:
def multiply_sparse(A, B):                     # A is R x K, B is K x C
    nz_B = [[(j, v) for j, v in enumerate(row) if v] for row in B]   # non-zeros of each row of B
    out = [[0] * len(B[0]) for _ in A]
    for i, row in enumerate(A):
        for k, a in enumerate(row):
            if a:                              # a zero in A meets nothing
                for j, b in nz_B[k]:
                    out[i][j] += a * b
    return out


def life_step(live):                           # live = a set of (r, c): no borders at all
    votes = Counter((r + dr, c + dc) for r, c in live
                    for dr in (-1, 0, 1) for dc in (-1, 0, 1) if dr or dc)
    return {cell for cell, k in votes.items() if k == 3 or (k == 2 and cell in live)}


print(multiply_sparse([[1, 0, 0], [-1, 0, 3]], [[7, 0, 0], [0, 0, 0], [0, 0, 1]]))   # [[7, 0, 0], [-7, 0, 3]]
print(sorted(life_step({(1, 0), (1, 1), (1, 2)})))   # [(0, 1), (1, 1), (2, 1)]

**Try it**
- Count the multiplications with a counter in the innermost loop: 3 for this example, against 2 · 3 · 3 = 18 for the plain triple loop.
- Run `life_step` twice on the blinker: you are back at `{(1, 0), (1, 1), (1, 2)}`.
- A glider walks: run `life_step` four times on `{(0, 1), (1, 2), (2, 0), (2, 1), (2, 2)}` and every cell has moved one step down and one step right.

### Say it in the interview

The spiral is the one you are most likely to say out loud, so here it is as a script.

> "First the brute force: a walker with a visited set that turns right whenever it's blocked. That's O(R·C) time and O(R·C) extra space. The visited set is the waste: the unvisited cells always form a rectangle, so four walls describe them.
>
> I walk the top row, the right column, the bottom row and the left column, moving each wall in right after I walk it, and I re-check before the last two sides because the rectangle can shrink to a single row or column. O(R·C) time, O(1) extra space."

Then point at the two re-checks, and test a 1 x 3, a 3 x 1 and one non-square grid out loud. For the other tools, the line to point at is the one that makes in-place safe: `range(r + 1, n)` in the transpose, "flags first, borders last" in Set Matrix Zeroes, `& 1` in Game of Life.

### Problem map

| Problem | Where | Key insight |
|---|---|---|
| Game of Life | `practice/simple/basics/matrices/05_game_of_life_in_place.py` | today in bit 0, tomorrow in bit 1; neighbours read `cell & 1`; shift everything at the end |
| Rotate Image | `math_geometry/rotate_image.py` · `practice/simple/basics/matrices/01_rotate_image.py` | clockwise = transpose (swap only above the diagonal) + reverse each row |
| Search a 2D Matrix II | `practice/simple/basics/matrices/04_search_2d_matrix_staircase.py` | start top-right: too big drops a column, too small drops a row; O(R + C) |
| Set Matrix Zeroes | `math_geometry/set_matrix_zeroes.py` · `practice/simple/basics/matrices/03_set_matrix_zeroes.py` | row 0 and column 0 hold the flags; remember their own zeros first, zero them last |
| Spiral Matrix | `math_geometry/spiral_matrix.py` · `practice/simple/basics/matrices/02_spiral_matrix.py` | four inclusive walls; walk a side, move its wall in; re-check before the bottom and left sides |

### Self-check

1. Why does the spiral need `if top <= bottom` before the bottom row, but no check before the right column?
<details><summary>Answer</summary>After <code>top += 1</code> there may be no rows left. The right-column loop is then simply empty (its <code>range(top, bottom + 1)</code> has nothing in it), but the bottom-row loop would walk the row we just walked, backwards. The <code>if</code> stops that. The same reasoning puts <code>if left &lt;= right</code> before the left column.</details>

2. In Set Matrix Zeroes, why is it safe to overwrite `m[r][0]` with a flag?
<details><summary>Answer</summary>We only write it when row r contains a zero, and then the whole row, including <code>m[r][0]</code>, ends up zero anyway. The one fact that writing destroys is whether column 0 itself held a zero, and that was saved in <code>col0</code> before marking began.</details>

3. Which corners can the staircase search start from, and why not the top-left?
<details><summary>Answer</summary>Top-right or bottom-left: each is the largest value of one line and the smallest of the other, so every comparison rules out a whole row or column. From the top-left both moves go to bigger values, so when the value is too small you don't know which way to go.</details>

4. Where does `(r, c)` go in a clockwise turn of an R x C grid, and how do you check it?
<details><summary>Answer</summary>To <code>(c, R-1-r)</code>, in a new C x R grid. Check two cells: <code>(0, 0)</code> must land in the top-right corner, <code>(0, R-1)</code>, and <code>(0, 1)</code> must land one row below it, <code>(1, R-1)</code>. Then try the map on a non-square grid.</details>

<a id="s22"></a>

## Math, Bits & Geometry

> An integer is a row of bits, and a row of bits is a small set: bit `v` is 1 exactly when `v` is in it. Bits answer set questions a whole word at a time; the math tools count things instead of listing them.

[Matrices](#s21) turned a grid into index arithmetic on `(r, c)`. This section does the same arithmetic inside one integer, whose bits are a row of cells you can set, flip and count, and then on the digits, primes and points behind the math questions.

**Reach for it when** the problem mentions *bits*, *XOR*, *powers of two*, *"every element appears twice except one"*, *subsets of a small set* (n ≤ 20, 26 letters, 10 digits), *parity* ("an odd number of times"), *modulo 10⁹ + 7*, *gcd, primes or digits*, *points, lines and slopes*, or when n goes up to 10⁹ so nothing can be listed one at a time.

**In this repo:** `bit_manipulation/` (9 problems) · `math_geometry/` (7 here; Rotate Image, Spiral Matrix and Set Matrix Zeroes are in [Matrices](#s21)) · basics: `practice/simple/basics/bits/01_get_set_clear_toggle_bits.py`, `02_count_bits_and_lowest_set_bit.py`, `03_xor_tricks_single_number_missing_number.py`, `04_bitmask_subset_enumeration.py`, `05_reverse_bits_and_shifts.py` and `practice/simple/basics/math/01_gcd_lcm_euclid.py`, `02_primes_sieve_and_primality.py`, `03_fast_power_and_modular_arithmetic.py`, `04_digits_reverse_integer_and_palindrome_number.py`, `05_base_conversion_and_excel_columns.py`, `06_random_pick_and_reservoir_sampling.py` · a bitset with a lazy global flip: `design/design_bitset.py`, in [Design Problems](11_Design.ipynb#s24)

### The picture

```text
   x = 180 =   1 0 1 1 0 1 0 0          bit i is worth 2^i; bit 0 is on the right
       bit:    7 6 5 4 3 2 1 0
   as a set:   {2, 4, 5, 7}             bit v is 1  <=>  v is in the set

          x  =  1 0 1 1 0 1 0 0
      x - 1  =  1 0 1 1 0 0 1 1         the borrow turns the lowest 1 into 0 and the 0s below it into 1s
  x & (x-1)  =  1 0 1 1 0 0 0 0         -> the lowest 1 is deleted    (repeat until 0 to count the 1s)
         -x  =  0 1 0 0 1 1 0 0         -x = ~x + 1 (8 bits shown): every bit above the lowest 1 flips
     x & -x  =  0 0 0 0 0 1 0 0         -> only the lowest 1 is kept  (= 4)

  XOR adds each column and keeps only "odd or even":  a ^ a = 0,  a ^ 0 = a,  in any order
       4 = 1 0 0
       1 = 0 0 1          4 ^ 1 ^ 2 ^ 1 ^ 2: in every column the pairs cancel,
       2 = 0 1 0          and the parity row that is left, 1 0 0, is the loner 4
       1 = 0 0 1
       2 = 0 1 0
     xor = 1 0 0
```

A set of small integers, such as letters, digits or up to about 60 items, fits in one word, so "add", "is it in?", union, intersection, subset test and "flip the parity" are each one operation instead of a loop. The math tools win the same way, by **counting instead of listing**: the sieve crosses off multiples instead of testing every number, fast power squares instead of multiplying n times, and the Hard puzzles at the end count a whole digit column, subtree or block of permutations at a time.

### From idea to code

Two parts follow, **bits** and then **math recipes**; the Hard puzzles that reuse them wait at the end of Variations. The bits rest on one template, the one "bit array" interview questions are built on: a set of small integers stored as bits. The idea in one sentence: *to remember which small integers you have seen, keep one bit per possible value: test bit `v` to ask, set bit `v` to add.*

The **State** is one integer, `seen`, used as a set, and its **Definition** is that `(seen >> v) & 1` is 1 exactly when value `v` has appeared so far. The **Invariant** follows: after reading `nums[i]`, `seen` holds exactly the values in `nums[0..i]`. **Init** is `seen = 0`, the empty set.

A **Step** tests bit `v` and only *then* adds it, `seen |= 1 << v`: the test asks whether v appeared *before* this item, so it must run while `seen` still describes the earlier items. The **Record** happens when the test finds the bit already set, a repeat. The **Return** reads what is missing from the bits still 0 at the end, where `full & ~seen` is "everything not seen" in one step.

The template solves Set Mismatch: a list should hold 1..n once each, but one number was copied over another, so one value appears twice and one is missing; return both. `[1, 2, 2, 4]` gives `[2, 3]`. The set {1, ..., n} is `(1 << (n + 1)) - 2`, the bits 0..n minus bit 0, and `full & ~seen` keeps the single value never seen. A lone bit turns into its value with `.bit_length() - 1`.

In [ ]:
def find_error_nums(nums):                     # Set Mismatch: 1..n, one value twice, one missing
    n, seen, dup = len(nums), 0, -1            # STATE: bit v of seen = "v has appeared"; INIT: empty
    for v in nums:
        if (seen >> v) & 1:                    # test FIRST: is v already in the set?
            dup = v                            # RECORD: this is the second copy
        seen |= 1 << v                         # STEP: add v, after the test
    full = (1 << (n + 1)) - 2                  # the set {1, ..., n}
    missing = (full & ~seen).bit_length() - 1  # RECORD: the one bit of full that is still 0
    return [dup, missing]                      # RETURN


print(find_error_nums([1, 2, 2, 4]))           # [2, 3]
print(find_error_nums([3, 1, 3]))              # [3, 2]

**Try it**
- Print `bin(full)`, `bin(seen)` and `bin(full & ~seen)` for `[1, 2, 2, 4]`: `0b11110`, `0b10110`, `0b1000`. A set difference is one `&`.
- Swap the test and the add (add first): every value now looks like a repeat, and `[1, 2, 2, 4]` answers `[4, 3]`.
- Run `find_error_nums([2, 2])`: `[2, 1]`. The hole can be the smallest value too.

How big can the set be? An int bitset costs O(largest value / 64) per operation: perfect for letters, digits and n up to about 10⁴; beyond that, or whenever you claim O(1), use a bytearray or 64-bit words. A Python int is immutable, so every `|=` builds a new one. The capstone at the end of the bit toolbox builds the 64-bit-word version.

### Watch it work

The trace runs the template on `[3, 1, 4, 1, 5, 2]` and draws the set after each value as 8 bits, value 0 on the right. The second 1 finds its bit already set, so the test fires before the add changes anything. The last two lines are the full set {1, ..., 6} and what `seen` lacks of it: one bit, the missing value 6.

In [ ]:
def trace_seen(nums):
    seen = 0
    for v in nums:
        note = "   <- bit already set: repeat" if (seen >> v) & 1 else ""
        seen |= 1 << v
        print(f"add {v}:   {seen:08b}{note}")
    full = (1 << (len(nums) + 1)) - 2
    print(f"full:    {full:08b}")
    print(f"missing: {full & ~seen:08b}")


trace_seen([3, 1, 4, 1, 5, 2])

**Try it**
- Run `trace_seen([2, 2, 1])` and read the missing value off the last line: bit 3, so 3.
- Add `seen.bit_count()` to the printed line: it grows by one per new value and stands still on the repeat.
- Change `- 2` to `- 1` in `full`: the missing line becomes `01000001`. Bit 0 joins the real hole, because the values start at 1 and bit 0 is never set.

### The bit toolbox

The template used three moves on single bits. The toolbox adds the rest, each piece building on the one before: the tricks to type without thinking, XOR as parity, Python's negative numbers, a greedy that reads the lowest bits, and a bit array that scales.

`1 << i` is a lone 1 at position i: OR it in to set bit i, AND with its complement to clear it, XOR to flip it, and shift down and `& 1` to read it. On whole sets, `a | b`, `a & b` and `a & ~b` are union, intersection and difference, and a is a subset of b exactly when `(a & ~b) == 0`. `for mask in range(1 << n)` visits every subset of n items, bit i saying whether item i is in.

The lowest 1 has tricks of its own. `x & (x - 1)` deletes it, so a loop of it runs once per 1-bit, and a power of two, with exactly one 1, becomes 0; so does 0 itself, so test `x > 0` too. `x & -x` keeps only the lowest 1, which makes `(x & -x).bit_length() - 1` the smallest member, and `~x & (x + 1)` keeps the lowest *0* as a lone 1. The size of the set is `x.bit_count()`.

The cell tries each trick on 180, the set {2, 4, 5, 7}, lists the eight subsets of `"abc"`, then solves three problems. Number of 1 Bits counts the 1s of an integer, 4 for 180, with one `x &= x - 1` per 1-bit. Counting Bits asks for that count for every i from 0 to n, `[0, 1, 1, 2, 1, 2]` for n = 5, and since `i >> 1` is `i` without its last bit, each answer reuses an earlier one. Reverse Bits mirrors a 32-bit integer, so 1 becomes 2³¹ = 2147483648.

In [ ]:
def show(x, width=8):
    return format(x, f"0{width}b")             # x in binary, padded to width digits


x = 0b10110100                                 # 180, the set {2, 4, 5, 7}
print((x >> 5) & 1, (x >> 3) & 1)              # 1 0: get bit 5, get bit 3
print(show(x | (1 << 3)), show(x & ~(1 << 2)), show(x ^ (1 << 7)))   # 10111100 10110000 00110100
print(show(x & (x - 1)), show(x & -x))         # 10110000 00000100: delete / keep the lowest 1
y = 0b10110111
print(show(~y & (y + 1)))                      # 00001000: the lowest 0 of y, as a lone 1
print([[ch for i, ch in enumerate("abc") if (mask >> i) & 1] for mask in range(1 << 3)])
# [[], ['a'], ['b'], ['a', 'b'], ['c'], ['a', 'c'], ['b', 'c'], ['a', 'b', 'c']]


def popcount(x):                               # Number of 1 Bits: one round per 1-bit
    count = 0
    while x:
        x &= x - 1                             # delete the lowest 1
        count += 1
    return count


def count_bits_upto(n):                        # Counting Bits: popcount of every i in 0..n
    ans = [0] * (n + 1)
    for i in range(1, n + 1):
        ans[i] = ans[i >> 1] + (i & 1)         # i is its half (i >> 1) with one bit appended
    return ans


def reverse_bits(x):                           # Reverse Bits: exactly 32 rounds
    out = 0
    for _ in range(32):
        out = (out << 1) | (x & 1)             # push x's lowest bit onto the right of out
        x >>= 1
    return out


print(popcount(180), (180).bit_count())                        # 4 4
print([v for v in range(20) if v > 0 and (v & (v - 1)) == 0])  # [1, 2, 4, 8, 16]: exactly one 1-bit
print(count_bits_upto(5))                                      # [0, 1, 1, 2, 1, 2]
print(reverse_bits(1), reverse_bits(43261596))                 # 2147483648 964176192

**Try it**
- Drop the `v > 0 and` from the power-of-two test: 0 joins the list, because `0 & (0 - 1)` is `0 & -1`, which is 0.
- Replace `for _ in range(32):` with `while x:` in `reverse_bits`: `reverse_bits(1)` returns 1 instead of 2147483648. The 31 leading zeros of the input never became trailing zeros of the output.
- Print `count_bits_upto(15)`: the second half (8..15) is the first half (0..7) plus one, because those numbers are the first half with bit 3 added.
- Change `"abc"` to `"abcd"` and `1 << 3` to `1 << 4` in the subsets line: 16 subsets, and the last 8 are the first 8 with `'d'` added.

XOR remembers, for every bit column, whether it has seen an odd or an even number of 1s. Values that come in pairs cancel, whatever their order, so what is left is the value with no partner. Single Number asks for the one value that appears once when every other value appears twice: `[4, 1, 2, 1, 2]` gives 4. Missing Number asks which value of 0..n is absent from n distinct numbers, `[3, 0, 1]` gives 2: XOR every index with every value, and each present value cancels against its equal index.

In [ ]:
def single_number(nums):                       # every value twice except one
    acc = 0
    for x in nums:
        acc ^= x                               # pairs cancel: a ^ a = 0
    return acc


def missing_number(nums):                      # 0..n with exactly one value missing
    acc = len(nums)                            # n has no index of its own: start with it
    for i, x in enumerate(nums):
        acc ^= i ^ x                           # a present value cancels against its equal index
    return acc


print(single_number([4, 1, 2, 1, 2]), missing_number([3, 0, 1]))   # 4 2

**Try it**
- Run `single_number([2, 2, 3, 2])`: 1, which is neither value. A value that appears three times no longer cancels; count each column modulo 3 instead, as Single Number II does in the next cell.
- Start `missing_number` with `acc = 0` and run `missing_number([0, 1])`: 0 instead of 2. The value n never got a partner.
- Single Number III (260) has two loners: `both = single_number([1, 2, 1, 3, 2, 5])` is `3 ^ 5 = 6`, and `low = both & -both` is 2, a bit where 3 and 5 differ. XOR-ing only the numbers that have that bit, `single_number([v for v in [1, 2, 1, 3, 2, 5] if v & low])`, gives 3, and `both ^ 3` is 5.

A Python int has no width. A negative number behaves as if its sign bit repeated forever to the left, so `-5` is `...1111 1011` with infinitely many 1s. The operators work on that infinite pattern, which is why `x & -x` is right for negatives too, but also why `bin(-5)` cannot show the bits, why `-1 >> 1` stays `-1`, and why a loop that waits for the bits to run out never ends. When a problem means 32-bit integers, mask to 32 bits, work there, and convert back at the end.

```text
   5 = ...0000 0101          no width: the sign bit repeats forever to the left
  -5 = ...1111 1011          bin(-5) hides this and prints '-0b101'
  -5 & 0xFF       = 1111 1011 = 251               keep the low 8 bits
  -5 & 0xFFFFFFFF = 4294967291                    the 32-bit pattern C and Java store
  back to signed:  u - (1 << 32) if u >> 31 else u      (bit 31 set means negative)
```

Two problems need exactly that. Single Number II (137) has every value three times except one, `[-2, -2, 1, 1, 4, 1, 4, 4, -4, -2]` gives -4: count the 1s in each of the 32 bit columns modulo 3, and the columns left over spell the loner's pattern. Sum of Two Integers (371) adds without `+`: `a ^ b` is the sum without carries, `(a & b) << 1` is the carries, and the loop repeats until no carry is left, all inside 32 bits.

In [ ]:
def to_signed32(u):                            # a 32-bit pattern back to a Python int
    return u - (1 << 32) if u >> 31 else u     # bit 31 set: the pattern is negative


def single_number_ii(nums):                    # Single Number II (137): all values 3 times but one
    out = 0
    for b in range(32):
        if sum((x >> b) & 1 for x in nums) % 3:   # this column's count is not a multiple of 3
            out |= 1 << b                      # so the loner has a 1 here
    return to_signed32(out)


def get_sum(a, b):                             # Sum of Two Integers (371): add without +
    MASK = 0xFFFFFFFF
    a, b = a & MASK, b & MASK                  # work on the 32-bit patterns
    while b:
        a, b = a ^ b, ((a & b) << 1) & MASK    # the sum without carries, then the carries
    return to_signed32(a)


print(bin(-5), -5 & 0xFF, -5 & 0xFFFFFFFF)                    # -0b101 251 4294967291
print(single_number_ii([-2, -2, 1, 1, 4, 1, 4, 4, -4, -2]))  # -4
print(get_sum(2, 3), get_sum(-1, 1), get_sum(-2, -3))        # 5 0 -5

**Try it**
- Return `out` instead of `to_signed32(out)` in `single_number_ii`: the example prints 4294967292, the 32-bit pattern of -4.
- Remove every `& MASK` from `get_sum` and add a counter that breaks the loop after 100 rounds: `get_sum(-1, 1)` is still running when it stops, because the carry walks left through the infinitely many 1s of -1.
- Print `bin(-1)`, `show(-x)` and `show(-x & 0xFF)`: `'-0b1'`, `'-10110100'`, `'01001100'`. Python shows a minus sign until you mask. It is also why `popcount(-1)` would never finish: -1 has infinitely many 1s to delete.

A greedy can read the low bits too. When the moves are "halve" and "add or subtract 1", halving only drops the last bit, so the cost of reaching 1 lies in clearing 1-bits.

Integer Replacement asks for the fewest steps from n to 1, where an even n is halved and an odd n becomes n + 1 or n − 1: 7 takes 4 steps, 7 → 8 → 4 → 2 → 1. For an odd n the last two bits decide. Ending in `01`, subtract: that leaves `00` and two free halvings. Ending in `11`, add: the carry wipes out the whole run of 1s at once. The single exception is 3, where 3 → 2 → 1 beats 3 → 4 → 2 → 1.

In [ ]:
def integer_replacement(n):
    steps = 0
    while n > 1:
        if n % 2 == 0:
            n //= 2                            # even: halving is the only move
        elif n == 3 or (n & 3) == 1:           # ends in 01 (or is 3): subtract
            n -= 1
        else:                                  # ends in 11: add; the carry clears the run of 1s
            n += 1
        steps += 1
    return steps


print(integer_replacement(8), integer_replacement(7), integer_replacement(2**31 - 1))   # 3 4 32

**Try it**
- Print `bin(n)` at every step for 15: `0b1111 → 0b10000 → 0b1000 → 0b100 → 0b10 → 0b1`, 5 steps. One `+ 1` turned four 1s into a single 1.
- Remove `n == 3 or`: `integer_replacement(3)` becomes 3 instead of 2 (3 → 4 → 2 → 1).
- Compare with a brute force that tries both moves at every odd number, `f(n) = 1 + min(f(n + 1), f(n - 1))`, for n in 1..200: always equal.

The capstone keeps every operation O(1) for a big universe by cutting the bits into fixed-size words. Bit `i` lives in word `i >> 6`, which is `i // 64`, at position `i & 63`, which is `i % 64`. Set, clear and get touch one word. Counting is a popcount per word, and "the next member from `i` on" skips empty words 64 values at a time, then reads the lowest 1 with `x & -x`.

```text
          word 2                 word 1            word 0
  [ 191 ... 130 129 128 ]  [ 127 ... 65 64 ]  [ 63 ... 2 1 0 ]
            ^
            i = 130: word 130 >> 6 = 2, position 130 & 63 = 2        (bytes: byte i >> 3, bit i & 7)
```

`Bits(200)` holds a set of integers in [0, 200) in four words. After setting 3, 64, 130 and 199 it counts 4 members; `next_set(4)` skips the rest of word 0 and finds 64, `next_set(131)` finds 199, and `next_set(200)` is −1, because no member is 200 or more.

In [ ]:
class Bits:                                    # a set of integers in [0, n), stored as 64-bit words
    def __init__(self, n):
        self.words = [0] * ((n + 63) // 64)    # STATE: bit i lives in word i >> 6, position i & 63

    def set(self, i):   self.words[i >> 6] |= 1 << (i & 63)
    def clear(self, i): self.words[i >> 6] &= ~(1 << (i & 63))
    def get(self, i):   return (self.words[i >> 6] >> (i & 63)) & 1
    def count(self):    return sum(w.bit_count() for w in self.words)

    def next_set(self, i):                     # smallest member >= i, or -1
        w = i >> 6
        if w >= len(self.words):
            return -1
        word = self.words[w] & ~((1 << (i & 63)) - 1)   # forget the bits below i
        while not word:                        # skip empty words, 64 values at a time
            w += 1
            if w == len(self.words):
                return -1
            word = self.words[w]
        return (w << 6) + (word & -word).bit_length() - 1   # the lowest 1 left in that word


b = Bits(200)
for i in (3, 64, 130, 199):
    b.set(i)
print(b.count(), b.get(64), b.get(65))                                   # 4 1 0
print(b.next_set(0), b.next_set(4), b.next_set(131), b.next_set(200))    # 3 64 199 -1

**Try it**
- Drop the mask in `next_set` (use `word = self.words[w]`): `b.next_set(4)` returns 3, an index below 4.
- Flip every bit at once, `b.words = [~w & (2**64 - 1) for w in b.words]`, and call `b.count()`: 252, not 196. The last word also flipped its 56 padding bits, the values 200..255, which are not in the universe. [Design Problems](11_Design.ipynb#s24) handles "flip everything" lazily, with one flag and a maintained count.
- Time it (`import time` first): put `random.sample(range(10**6), 20_000)` through `Bits.set` and through a plain int (`seen |= 1 << v`), timing each with `time.perf_counter()`. The int is dozens of times slower, because every `|=` copies the whole number.

### Math recipes

The math recipes count instead of listing too. Four of them cover what medium interviews ask: number theory, digits and bases, random sampling and exact slopes.

Euclid's rule is that every common divisor of `a` and `b` also divides `a % b`, so `gcd(a, b) = gcd(b, a % b)`, and the pair shrinks fast; then `lcm(a, b) = a // gcd(a, b) * b`. Trial division stops at √n, because divisors come in pairs `d · (n / d)` and one of the two is at most √n. The sieve lists the primes up to n by crossing off the multiples of each prime p from `p * p` on; the smaller multiples have a smaller prime factor and are gone already.

Fast power writes the exponent in binary, `3^13 = 3^8 · 3^4 · 3^1`: square the base once per bit and multiply it in when the bit is 1. Reduce after every multiply, because `(a * b) % m == ((a % m) * (b % m)) % m`: reducing early never changes the answer and keeps every number below m², under 2⁶³ for m = 10⁹ + 7, which fixed-width languages need. Python never overflows, but `3 ** 10**6` has 477 122 digits, and multiplying giants is slow.

Python has most of this built in: `math.gcd`, `math.lcm`, `pow(b, e, m)`, `pow(x, -1, m)` for the inverse of x modulo m, `math.isqrt` and `math.comb`. Write each one once by hand so that you can explain it: gcd(252, 105) is 21, ten primes lie below 30, and 3¹³ mod 1000 is 323.

In [ ]:
def gcd(a, b):
    while b:
        a, b = b, a % b                        # (252, 105) -> (105, 42) -> (42, 21) -> (21, 0)
    return abs(a)


def lcm(a, b):
    return abs(a) // gcd(a, b) * abs(b) if a and b else 0     # divide first: smaller numbers


def is_prime(n):                               # trial division up to sqrt(n): a pair has one side there
    return n >= 2 and all(n % d for d in range(2, math.isqrt(n) + 1))


def primes_upto(n):                            # sieve of Eratosthenes
    alive = [False, False] + [True] * (n - 1)  # alive[i]: is i still possibly prime?
    for p in range(2, math.isqrt(n) + 1):
        if alive[p]:
            for multiple in range(p * p, n + 1, p):   # smaller multiples are crossed off already
                alive[multiple] = False
    return [i for i in range(n + 1) if alive[i]]


def power_mod(base, exp, mod):
    result, base = 1 % mod, base % mod
    while exp:
        if exp & 1:                            # this bit of exp is 1: multiply its power in
            result = result * base % mod       # reduce after EVERY multiply
        base = base * base % mod               # base, base^2, base^4, base^8, ...
        exp >>= 1
    return result


print(gcd(252, 105), lcm(4, 6), is_prime(91), primes_upto(30))   # 21 12 False [2, 3, 5, 7, 11, 13, 17, 19, 23, 29]
print(power_mod(3, 13, 1000), pow(3, 13, 1000))                  # 323 323

**Try it**
- Print `(a, b)` at the top of the `gcd` loop for `gcd(55, 34)`: 8 rounds. Neighbouring Fibonacci numbers are the slowest case for Euclid, and even they finish in O(log) steps.
- Remove the `1 % mod` (start with `result = 1`) and run `power_mod(5, 0, 1)`: 1 instead of 0. Everything is 0 modulo 1, even an empty product.
- Compare `power_mod(3, 10**6, 10**9 + 7)` with `3 ** 10**6 % (10**9 + 7)`: the same answer, but the second first builds a 1.6-million-bit number. The built-in `pow(3, 10**6, 10**9 + 7)` does the fast version for you.

Digits come next. `n % 10` is the last digit and `n // 10` drops it, so digits come out lowest first, and `rev * 10 + d` pushes a digit onto the right end of `rev`. Work on `abs(x)` and put the sign back at the end. Every base works the same way with `divmod(n, b)`, and reading digits back is Horner's rule, `n = n * b + d`. Excel columns are base 26 with digits 1..26 and *no zero*, so shift each digit down by one before the divmod.

Reverse Integer asks for the digits of a 32-bit integer in reverse order, or 0 when the result leaves 32 bits: −123 gives −321, and 1534236469 gives 0. A 32-bit language cannot compute `rev * 10 + d` and look afterwards, so test *before* pushing, `rev > (LIMIT - d) // 10`. One limit, 2³¹ − 1, serves both signs: −2³¹ could only come from reversing 8463847412, which is not a 32-bit input.

Palindrome Number asks whether an integer reads the same backwards, without turning it into a string: 12321 does, 10 does not. Reverse only the lower half of the digits and compare it with the upper half. Excel Sheet Column Title turns a column number into its letters, 28 into `AB`, and `column_number` goes back, `ZY` to 701.

In [ ]:
def reverse_int(x):                            # Reverse Integer: 0 if the result leaves 32 bits
    LIMIT = 2**31 - 1
    n, rev = abs(x), 0
    while n:
        n, d = divmod(n, 10)                   # peel off the last digit
        if rev > (LIMIT - d) // 10:            # rev * 10 + d would pass the limit: stop BEFORE
            return 0
        rev = rev * 10 + d                     # push d onto the right of rev
    return rev if x >= 0 else -rev


def is_palindrome_number(x):                   # Palindrome Number, without strings
    if x < 0 or (x % 10 == 0 and x != 0):      # a minus sign, or a trailing 0 that cannot lead
        return False
    rev = 0
    while x > rev:                             # move digits until rev holds the lower half
        x, d = divmod(x, 10)
        rev = rev * 10 + d
    return x == rev or x == rev // 10          # even length, or odd (the middle digit is in rev)


def column_title(n):                           # 1 -> A, 26 -> Z, 27 -> AA
    out = []
    while n:
        n, r = divmod(n - 1, 26)               # shift 1..26 down to 0..25 first
        out.append(chr(ord("A") + r))
    return "".join(reversed(out))


def column_number(title):
    n = 0
    for ch in title:
        n = n * 26 + (ord(ch) - ord("A") + 1)  # Horner: shift one digit left, add this one
    return n


print(reverse_int(-123), reverse_int(1534236469), reverse_int(120))     # -321 0 21
print(is_palindrome_number(12321), is_palindrome_number(1221), is_palindrome_number(10))   # True True False
print(column_title(28), column_title(701), column_number("ZY"))         # AB ZY 701

**Try it**
- Remove the `- 1` in `column_title` and run `column_title(26)`: `"BA"` instead of `"Z"`. Plain base 26 writes 26 as "10"; Excel has no zero digit, so 26 is the single digit Z.
- `reverse_int(1463847412)` is 2147483641, which fits; `reverse_int(1563847412)` would be 2147483651, which does not, so it returns 0.
- Drop the `x % 10 == 0 and x != 0` test and run `is_palindrome_number(10)`: `True`. Its trailing 0 becomes the leading digit of `rev`, and the comparison cannot see it.

The third recipe is random sampling. Reservoir sampling picks k items uniformly from a stream too long to store: keep the first k, and after that item i, counting from 0, replaces a random member with probability k/(i+1). With k = 1 it is the random pick behind Linked List Random Node, a uniformly random node of a list of unknown length, and Random Pick Index, a random index among the copies of a target value.

Why that is uniform: item i gets in with probability k/(i+1), and each later item j removes it with probability (k/(j+1)) · (1/k) = 1/(j+1), so it survives item j with probability j/(j+1). The product k/(i+1) · (i+1)/(i+2) · ... · (n−1)/n telescopes to k/n. The cell samples 2 of `range(5)` ten thousand times, so each value should be picked about 10 000 · 2/5 = 4000 times.

In [ ]:
def reservoir_sample(stream, k):
    sample = []
    for i, x in enumerate(stream):
        if i < k:
            sample.append(x)                   # the first k fill the reservoir
        else:
            j = random.randrange(i + 1)        # a uniform slot in 0..i
            if j < k:                          # happens with probability k / (i + 1)
                sample[j] = x                  # replace a uniformly chosen member
    return sample


random.seed(0)
hits = Counter()
for _ in range(10_000):
    hits.update(reservoir_sample(range(5), 2))
print(sorted(hits.items()))   # [(0, 3990), (1, 3927), (2, 4004), (3, 4035), (4, 4044)]: all near 4000

**Try it**
- Change `random.randrange(i + 1)` to `random.randrange(i)` and rerun the cell: values 2, 3 and 4 now show up about 5000 times and 0 and 1 about 2500. Item 2 always gets in, since `j` can only be 0 or 1.
- Use `k = 1` on `range(3)` for 9000 runs: each value about 3000 times. That is random pick.
- `reservoir_sample(range(3), 5)`: `[0, 1, 2]`. With fewer items than k, you simply get them all.

The last recipe is geometry without floats. Max Points on a Line asks for the largest number of the given points that lie on one straight line: `[[1, 1], [3, 2], [5, 3], [4, 1], [2, 3], [1, 4]]` gives 4. All points on one line through an anchor share a direction `(dx, dy)`. Floats cannot be trusted as keys, but the direction reduced by the gcd, with a fixed sign, is exact and hashable: `(2, 6)` and `(-1, -3)` both become `(1, 3)`, and vertical lines are `(0, 1)` with no division at all.

A copy of the anchor has no direction, since `gcd(0, 0)` is 0, but it lies on every line through the anchor, so the code counts copies separately and adds them to the best line. Each point anchors only the points after it, because a line through an earlier point was already counted from there. Two copies of `[1, 1]` and the point `[2, 2]` make one line of 3.

In [ ]:
def slope_key(dx, dy):                         # one exact key per line direction
    g = math.gcd(dx, dy)                       # > 0 for two distinct points
    dx, dy = dx // g, dy // g
    if dx < 0 or (dx == 0 and dy < 0):         # (1, 2) and (-1, -2) are the same line
        dx, dy = -dx, -dy
    return dx, dy


def max_points(points):
    best = 0
    for i, (x1, y1) in enumerate(points):      # anchor every line at its first point
        same, dirs = 1, Counter()              # the anchor and its copies; directions to the rest
        for x2, y2 in points[i + 1:]:
            if (x1, y1) == (x2, y2):
                same += 1                      # a copy lies on every line through the anchor
            else:
                dirs[slope_key(x2 - x1, y2 - y1)] += 1
        best = max(best, same + max(dirs.values(), default=0))
    return best


print(slope_key(2, 6), slope_key(-1, -3), slope_key(0, -5))           # (1, 3) (1, 3) (0, 1)
print(max_points([[1, 1], [3, 2], [5, 3], [4, 1], [2, 3], [1, 4]]))     # 4
print(max_points([[1, 1], [1, 1], [2, 2]]))                             # 3

**Try it**
- Use a float key instead, `(y2 - y1) / (x2 - x1) if x2 != x1 else math.inf`, and run `max_points([[0, 0], [94911151, 94911150], [94911152, 94911151]])`: 3. The answer is 2: the two slopes differ, but they round to the same float.
- Delete the sign rule and run `max_points([[0, 0], [1, 1], [-1, -1]])`: 2 instead of 3, because `(1, 1)` and `(-1, -1)` became different keys.
- Call `slope_key(0, 0)`: `ZeroDivisionError`. That is why copies of the anchor never reach it.

### Where it goes wrong

Most bit bugs come from Python ints having no width, and most math bugs from floats or from numbering that starts at 1.

1. **Operator precedence.** In Python `&`, `|` and `^` bind tighter than `==`, so `x & 1 == 0` works, unlike in C and Java. But `<<` and `>>` bind *looser* than `+` and `-`: `1 << k - 1` is `1 << (k - 1)`, and `x & mask + 1` is `x & (mask + 1)`. Parenthesise every bit expression.
2. **`~x` is negative in Python.** Ints have no fixed width, so `~0b1010 == -11`, not `0b0101`. For a k-bit complement use `x ^ ((1 << k) - 1)`.
3. **Negative numbers never run out of bits.** `-1 >> 1 == -1`, so `while x: x >>= 1` never ends for a negative x, and neither does the `x &= x - 1` loop. Mask first: `x & 0xFFFFFFFF` is the 32-bit pattern, and `to_signed32` turns it back.
4. **Digits of a negative number.** `divmod(-123, 10) == (-13, 7)`, so the last digit comes out as 7; and `divmod(-1, 10) == (-1, 9)`, so `while n:` never ends. Work on `abs(x)` and restore the sign; [Python Toolkit](01_Start_Here.ipynb#s02) has more on `%` and `//` with negatives.
5. **Stopping a fixed-width loop early.** Reverse Bits must run exactly 32 rounds; stopping when `x == 0` loses the leading zeros that should become trailing zeros, so `reverse_bits(1)` comes back as 1 instead of 2147483648.
6. **Zero passes the power-of-two test.** `0 & (0 - 1) == 0`, so write `x > 0 and (x & (x - 1)) == 0`.
7. **Floats for exact questions.** `math.log(243, 3) == 4.999999999999999`, and `int(math.sqrt(100000001**2 - 1))` is 100000001 while `math.isqrt` gives the true 100000000. Use integers: reduced `(dx, dy)` pairs, cross products, `math.isqrt`, repeated multiplication instead of logs.
8. **Reducing mod at the wrong moment.** Reduce after every multiply (`ans = ans * x % MOD`), but never before comparing: with `a = 10**9 + 8` and `b = 5`, `max(a % MOD, b % MOD)` picks `b`.
9. **Off by one in 1-indexed systems.** Excel columns need `n - 1` before the divmod, or `column_title(26)` gives `"BA"`; the k-th permutation needs the rank `k - 1`; and "primes less than n" is not "primes up to n".
10. **Testing after adding.** In the bit-array template, test bit v *before* you set it, or every value looks like a repeat: `[1, 2, 2, 4]` answers `[4, 3]`.
11. **Duplicate points.** `slope_key(0, 0)` divides by `gcd(0, 0) == 0`: count copies of the anchor separately.

### Edge cases to say out loud

Say these before coding, because each one lands on a guard above. 0 and 1 (no set bits, `is_prime(1)`) · negative numbers (infinite sign bits in Python) · the 32-bit limits (2³¹ − 1 and −2³¹) · the missing value is 0 or n · duplicate points and vertical lines · `mod = 1` · an empty input.

In [ ]:
assert find_error_nums([2, 2]) == [2, 1] and find_error_nums([1, 1]) == [1, 2]
assert popcount(0) == 0 and popcount(2**32 - 1) == 32
assert reverse_bits(0) == 0 and reverse_bits(2**32 - 1) == 2**32 - 1
assert single_number([-3, 7, -3]) == 7                      # negatives cancel too
assert missing_number([0]) == 1 and missing_number([1]) == 0
assert single_number_ii([-1, -1, -1, 7]) == 7 and get_sum(0, 0) == 0
assert Bits(1).count() == 0 and Bits(1).next_set(0) == -1
assert gcd(0, 5) == 5 and lcm(0, 5) == 0 and gcd(-12, 18) == 6
assert not is_prime(1) and is_prime(2) and primes_upto(1) == []
assert power_mod(5, 0, 1) == 0 and power_mod(-2, 3, 5) == 2  # with a positive modulus, % lands in [0, mod)
assert reverse_int(-2**31) == 0 and reverse_int(0) == 0 and reverse_int(120) == 21
assert is_palindrome_number(0) and not is_palindrome_number(-121)
assert column_title(1) == "A" and column_title(52) == "AZ" and column_number("AAA") == 703
assert max_points([]) == 0 and max_points([[0, 0]]) == 1 and max_points([[0, 0], [0, 0]]) == 2
assert max_points([[1, 0], [1, 5], [1, -3], [2, 2]]) == 3  # a vertical line
print("edge cases pass")

**Try it**
- Run `all(column_number(column_title(n)) == n for n in range(1, 1000))`: True. Two inverse functions test each other.
- What should `missing_number([])` return? Write the assert before running it (0: the range 0..0 lost its only value).
- Predict `find_error_nums([1, 2, 3, 3])` before running: `[3, 4]`, where the missing value is n itself, the top bit of `full`. Build `full` as `(1 << n) - 2` instead and that bit is lost: the answer becomes `[3, -1]`.

### Variations

The bit-array template is the root of a family: change what a bit means, or how masks are compared. The table says what changes and what each problem asks; most of the code is in the bit toolbox above, and the cell after the table works out the bit-column count.

| Variation | What changes from the template | Problems |
|---|---|---|
| **Parity set** | toggle instead of set (`^=`): bit d means "d has been seen an odd number of times" | 1371 Find the Longest Substring Containing Vowels in Even Counts: the longest substring with every vowel an even number of times, in [Prefix Sums](02_Arrays_Hashing_Prefix_Sums.ipynb#s04) |
| **Prefix masks + hash map** | a substring's mask is the XOR of two prefix masks; a `Counter` of masks counts the substrings | 1915 Number of Wonderful Substrings: how many substrings have at most one letter an odd number of times |
| **Letter sets** | a word becomes a 26-bit mask; two words share no letter exactly when `(a & b) == 0` | 318 Maximum Product of Word Lengths: the largest `len(a) * len(b)` over two words with no letter in common |
| **All subsets of n items** | `for mask in range(1 << n)`: bit i says whether item i is in | 78 Subsets: every subset of a list of distinct numbers |
| **Maximum XOR** | a binary trie of the numbers; from the top bit down, take the opposite bit whenever it exists | 421 Maximum XOR of Two Numbers in an Array: the largest `a ^ b` over all pairs |
| **XOR as parity** | pairs cancel: the single number, the missing number, two singles | 136 Single Number, 268 Missing Number, 260 Single Number III: the two values that appear once |
| **Count each bit column** | `% 3` per column for "three times" (137); `ones * (n - ones)` differing pairs per column (477) | 137 Single Number II; 477 Total Hamming Distance: the bit differences summed over all pairs, worked out below |
| **32-bit arithmetic** | mask with `0xFFFFFFFF`, convert back through bit 31 | 371 Sum of Two Integers, 190 Reverse Bits |
| **AND of a whole range** | the answer is the common binary prefix of the two ends: `while right > left: right &= right - 1` | 201 Bitwise AND of Numbers Range: the AND of every integer from `left` to `right` |
| **Gray code** | `i ^ (i >> 1)`: neighbours differ in exactly one bit | 89 Gray Code: all n-bit numbers in an order where neighbours differ in one bit |
| **Rolling bit code** | 2 bits per DNA letter, keep the last 20 bits with `& 0xFFFFF` | 187 Repeated DNA Sequences: every 10-letter stretch that occurs more than once |
| **Big universe** | a bytearray or 64-bit words | (no repo problem) |
| **Global flip** | a lazy flag and a maintained count, in [Design Problems](11_Design.ipynb#s24) | 2166 Design Bitset: fix, unfix, flip all and count, each in O(1) |
| **Long division** | a remainder seen before starts the repeating part; handle the sign with `abs` | 166 Fraction to Recurring Decimal: a fraction as a decimal string, the repeating part in parentheses |
| **Simulate one cycle** | after one pass of the moves, bounded iff back at the origin or not facing north | 1041 Robot Bounded In Circle: does a robot that repeats its moves forever stay inside a circle |
| *Second pass:* **Letter subsets** | flip the loop: walk the puzzle's 2⁷ submasks with `sub = (sub - 1) & mask` | 1178 Number of Valid Words for Each Puzzle: for each puzzle, how many words use only its letters and include its first letter |
| *Second pass:* **One odd digit** | earliest index of each prefix mask; look up the mask and its 10 one-bit neighbours | 1542 Find Longest Awesome Substring: the longest substring that can be rearranged into a palindrome |
| *Second pass:* **Maximum XOR with a limit** | sort the queries by limit and grow the binary trie offline | 1707 Maximum XOR With an Element From Array: the best `x ^ y` with y at most a per-query limit |
| *Second pass:* **Turns and corners** | the cross product's sign; toggled corners plus an area check | 587 Erect the Fence: the trees on the convex hull; 391 Perfect Rectangle: do rectangles tile a rectangle exactly |
| *Second pass:* **Count whole blocks** | a digit column, a subtree of prefixes, a block of (n−1)! permutations | 233 Number of Digit One: the 1s written in 1..n; 440 K-th Smallest in Lexicographical Order: the k-th of 1..n in dictionary order; 60 Permutation Sequence: the k-th ordering of 1..n |
| *Second pass:* **Outcomes, not tests** | a pig has T + 1 outcomes, so p pigs tell (T + 1)^p buckets apart | 458 Poor Pigs: the fewest pigs that find the one poisoned bucket |

Total Hamming Distance (477) asks for the number of differing bits summed over all pairs: `[4, 14, 2]` gives 6. The total is a sum over bit columns, and in a column with `ones` ones and `n - ones` zeros exactly `ones * (n - ones)` pairs differ. That is 32 passes instead of n² pairs.

In [ ]:
def total_hamming_distance(nums):              # Total Hamming Distance (477)
    n, total = len(nums), 0
    for b in range(32):                        # one bit column at a time
        ones = sum((x >> b) & 1 for x in nums)
        total += ones * (n - ones)             # each (1, 0) pair differs in this column
    return total


print(total_hamming_distance([4, 14, 2]))      # 6

**Try it**
- Check it against all pairs: `sum(bin(a ^ b).count("1") for a, b in combinations([4, 14, 2], 2))` is also 6.
- Print `ones` for every column of `[4, 14, 2]`: bits 1, 2 and 3 have 2, 2 and 1 ones, so the total is 2·1 + 2·1 + 1·2 = 6.
- Change `range(32)` to `range(3)`: the answer drops to 4. Bit 3 is never looked at, so its two differences, 14 against 4 and 14 against 2, are lost.

The rest of this section is a second pass: Hard problems that reuse the same moves. Skip them until the main path is automatic.

Number of Valid Words for Each Puzzle (1178) asks, for each 7-letter puzzle, how many words are valid for it: every letter of the word is in the puzzle, and the word contains the puzzle's first letter. Words with the same letter *set* are interchangeable, so a `Counter` holds each word's mask once. Then flip the loop: a puzzle has only 2⁷ = 128 letter subsets, so look each one up instead of testing every word. The walk over the subsets of a mask is one line, `sub = (sub - 1) & mask`:

```text
  mask      1 0 1 1      the allowed positions (the puzzle's letters)
  sub       1 0 0 0
  sub - 1   0 1 1 1      the borrow: the lowest 1 becomes 0, everything below it becomes 1
  & mask    0 0 1 1      drop the forbidden positions: the next smaller subset of mask
  the walk: 1011, 1010, 1001, 1000, 0011, 0010, 0001, 0000   (counting down on mask's own bits)
```

The cell prints that walk for `0b1011`, then counts the valid words for six puzzles. For `"actresz"` four words fit: `"aaaa"`, `"asas"`, `"actt"` and `"access"`. `"gaswxyz"` gets none, because no word contains its first letter, g.

In [ ]:
def letter_mask(word):
    m = 0
    for ch in word:
        m |= 1 << (ord(ch) - ord("a"))         # bit 0 = 'a', ..., bit 25 = 'z'
    return m


def submasks(mask):                            # every subset of mask, largest first, 0 last
    sub = mask
    while True:
        yield sub
        if sub == 0:
            return
        sub = (sub - 1) & mask                 # the next smaller subset


def valid_word_counts(words, puzzles):
    count = Counter(letter_mask(w) for w in words)
    out = []
    for p in puzzles:
        first = 1 << (ord(p[0]) - ord("a"))
        out.append(sum(count[s] for s in submasks(letter_mask(p)) if s & first))
    return out


print([bin(s) for s in submasks(0b1011)])   # ['0b1011', '0b1010', '0b1001', '0b1000', '0b11', '0b10', '0b1', '0b0']
print(valid_word_counts(["aaaa", "asas", "able", "ability", "actt", "actor", "access"],
                        ["aboveyz", "abrodyz", "abslute", "absoryz", "actresz", "gaswxyz"]))   # [1, 1, 3, 2, 4, 0]

**Try it**
- Replace `(sub - 1) & mask` with `sub - 1` and comment out the puzzle line before rerunning, because a puzzle with a `z` would now count down through tens of millions of numbers. The first line shows why: all 12 numbers from 11 down to 0, including ones like `0b100` that are not subsets.
- Move the `if sub == 0: return` check above the `yield`: the empty set 0 is never produced (7 values instead of 8).
- Drop the `if s & first` filter: the last puzzle, `"gaswxyz"`, answers 2 instead of 0, because `"aaaa"` and `"asas"` fit its letters but lack its first letter `g`.

Find Longest Awesome Substring (1542) asks for the longest substring of digits that can be rearranged into a palindrome: `"3242415"` gives 5, since `"24241"` rearranges into `"24142"`. It is the parity mask of [Prefix Sums](02_Arrays_Hashing_Prefix_Sums.ipynb#s04) with 10 bits, one per digit: keep the earliest index of each prefix mask, and at each end make 11 lookups, the same mask for all counts even and the 10 masks one bit away for one odd digit.

Maximum XOR With an Element From Array (1707) asks, for each query `(x, m)`, for the largest `x ^ y` over the numbers y ≤ m, or −1 if there is none: with the numbers 0..4, `(5, 6)` gives `5 ^ 2 = 7`. Bits are decided from the top, since a 1 at bit 29 outweighs all the lower bits together. So the numbers go into a binary trie, the trie of [Tries](06_Trees_Tries.ipynb#s12) with children 0 and 1, and a query walks down taking the child opposite to x's bit whenever it exists: 30 steps.

The limit is handled offline, which means answering the queries in an order of your choosing rather than the order given. Sort the queries by m and insert the sorted numbers while they are ≤ m, so the trie holds exactly the allowed numbers when each query is answered, and write every answer back to its query's original position. Answered in input order, a query would see the numbers above its limit that an earlier query inserted.

Erect the Fence (587) asks which trees stand on the convex hull, the tightest fence around them all, counting trees on a straight stretch: of `[[1, 1], [2, 2], [2, 0], [2, 4], [3, 3], [4, 2]]` all but (2, 2) do. Walk the sorted trees left to right with the lower fence on a stack, popping the last post while the path through it turns right, then walk back for the upper fence. Pop only on `cross < 0`: popping when straight deletes the trees on a straight stretch.

```text
   cross(o, a, b) = (a.x - o.x) * (b.y - o.y) - (a.y - o.y) * (b.x - o.x)

   cross(o, a, b) > 0       cross(o, a, b) < 0       cross(o, a, b) == 0
           b                  o ---- a                 o ---- a ---- b
           |                         |
     o --- a                         b
       left turn               right turn              straight
```

Perfect Rectangle (391) asks whether some axis-aligned rectangles tile one big rectangle with no gap and no overlap. They do exactly when their areas add up to the bounding box's area and toggling every tile's four corners in a set leaves only the box's four corners: every inner corner belongs to 2 or 4 tiles and switches back off. Both tests are needed: in `[[0, 0, 1, 1], [0, 0, 1, 1], [0, 0, 2, 2]]` the two copies of the small square cancel each other's corners, and only the area test sees the overlap.

```text
   X--------+-----X         A, B, C, D tile the box: every "+" is a corner of exactly 2 tiles,
   |   A    |  B  |         so toggling all corners switches every "+" back off and leaves
   +----+---+-----+         exactly the four X corners on
   |  C |    D    |
   X----+---------X
```

Number of Digit One (233) counts the 1s written in all the numbers from 1 to n: up to 13 there are 6, in 1, 10, 11 twice, 12 and 13. Count per column instead: in the column worth p the digit cycles through 0..9, holding each value for p numbers. Split n into `high`, `cur` and `low`, the digits above, at and below the column. Each of the `high` full cycles shows a 1 for p numbers, and the last cycle adds p if `cur > 1`, `low + 1` if `cur == 1`, else nothing.

```text
  n = 2315, tens column (p = 10): high = 23, cur = 1, low = 5
  0000..2299: 23 full cycles, each with ten numbers showing a 1 in the tens column -> 23 * 10 = 230
  2300..2315: we are inside the 1-run (cur = 1): 2310..2315                         -> low + 1 = 6
```

K-th Smallest in Lexicographical Order (440) asks for the k-th number of 1..n in dictionary order: for n = 13, k = 2 gives 10 and k = 6 gives 2. That order is a pre-order walk of the 10-ary tree whose node v has the children 10v..10v+9, drawn below. Standing on a node, count its subtree level by level, clipped to n. If the k-th number lies past that whole subtree, jump to the next sibling in one step; otherwise step down to the first child.

```text
  n = 13:     1          2   3   4   5   6   7   8   9         pre-order: 1, 10, 11, 12, 13, 2, ..., 9
           /  |  \  \
         10  11  12  13                                         subtree of 1 = {1} + {10..13} = 5 numbers
```

Permutation Sequence (60) asks for the k-th of the n! orderings of 1..n in lexicographic order: n = 3, k = 3 gives `"213"`. The orderings come in n blocks of (n−1)!, one per first digit, so the 0-based rank divided by (n−1)! picks the first digit from the unused ones, and the remainder is the rank inside that block; repeat with (n−2)!. That is the rank written in the factorial number system. The rank is `k - 1`: starting from k itself, `(3, 3)` gives `"231"`, the 4th ordering.

```text
  n = 4, k = 9 -> rank r = 8 = 1·3! + 1·2! + 0·1! + 0·0!
  unused [1 2 3 4] take index 1 -> 2;  [1 3 4] take index 1 -> 3;  [1 4] take index 0 -> 1;  [4] -> 4:  "2314"
```

Poor Pigs (458) asks for the fewest pigs that find the poisoned bucket when a pig dies `minutesToDie` after drinking and the test lasts `minutesToTest`: 4 buckets and one round need 2 pigs. Over T rounds a pig has T + 1 outcomes, death in one round or survival, so p pigs tell apart (T + 1)^p buckets. Label the buckets with p digits in base T + 1, and pig i drinks, in round r, every bucket whose i-th digit is r. So 1000 buckets and 4 rounds need 5 pigs: 5⁵ ≥ 1000 > 5⁴.

```text
  4 buckets, 1 round: a pig ends dead or alive, so 2 pigs tell apart 2 x 2 = 4 buckets
  bucket   label (A, B)      A drinks every bucket whose A-digit is 1: buckets 2 and 3
     0         0 0           B drinks every bucket whose B-digit is 1: buckets 1 and 3
     1         0 1
     2         1 0           A dies, B lives -> label 1 0 -> bucket 2 is the poisoned one
     3         1 1
```

### Say it in the interview

> "I need to remember which of 1..n I've seen, so I keep one bit per value. I test the bit before setting it; a bit already set means this value is the duplicate. At the end, the one bit of 1..n still 0 is the missing value. The brute force rescans for each value, O(n²). With a fixed-size bit array, a bytearray or 64-bit words, test and set touch one word, so it's O(n) time and n bits; a single Python int works too, but every `|=` copies it. If you want O(1) extra space, I'd mark value v by negating `nums[v - 1]`."

Point at the line that tests *before* it sets, and say what a bit means out loud: "bit v of `seen` means v appeared". The negation trick is in [Arrays & Hashing](02_Arrays_Hashing_Prefix_Sums.ipynb#s03). For the math tools, name the constraint that rules out listing, such as n up to 10⁹, then the unit you count instead: a bit of the exponent, a digit column, a subtree, a block of (n−1)!.

### Problem map

| Problem | Where | Key insight |
|---|---|---|
| Counting Bits | `bit_manipulation/counting_bits.py` | `ans[i] = ans[i >> 1] + (i & 1)`: i is its half with one bit appended |
| Erect the Fence | `math_geometry/erect_the_fence.py` | sort, build lower and upper chains on a stack; pop only on a right turn (cross < 0) so edge trees stay |
| Excel Sheet Column Title | `practice/simple/basics/math/05_base_conversion_and_excel_columns.py` | base 26 with digits 1..26 and no zero: `divmod(n - 1, 26)` |
| Find Longest Awesome Substring | `bit_manipulation/find_longest_awesome_substring.py` | 10-bit prefix parity mask; earliest index of the same mask or of a mask one bit away |
| Integer Replacement | `bit_manipulation/integer_replacement.py` | odd n: subtract on ...01 (and for 3), add on ...11 so the carry clears the run of 1s |
| K-th Smallest in Lexicographical Order | `math_geometry/kth_smallest_in_lexicographical_order.py` | pre-order of the 10-ary tree; skip a prefix's whole subtree if k is past it, else step down |
| Max Points on a Line | `math_geometry/max_points_on_a_line.py` | per anchor, count gcd-reduced (dx, dy) with a fixed sign; copies of the anchor join every line |
| Maximum XOR With an Element From Array | `bit_manipulation/maximum_xor_with_an_element_from_array.py` | sort queries by limit, insert nums up to it into a binary trie, take the opposite bit greedily |
| Missing Number | `bit_manipulation/missing_number.py` | XOR all indices 0..n with all values: present values cancel, the missing one stays |
| Number of 1 Bits | `bit_manipulation/number_of_1_bits.py` | `n &= n - 1` deletes the lowest 1; count the deletions |
| Number of Digit One | `math_geometry/number_of_digit_one.py` | per column p: `high * p` full cycles, plus p, `low + 1` or 0 depending on the column's digit |
| Number of Valid Words for Each Puzzle | `bit_manipulation/number_of_valid_words_for_each_puzzle.py` | Counter of word letter-masks; sum over the puzzle's 128 submasks that contain its first letter |
| Palindrome Number | `practice/simple/basics/math/04_digits_reverse_integer_and_palindrome_number.py` | reverse only the lower half of the digits and compare it with the upper half |
| Perfect Rectangle | `math_geometry/perfect_rectangle.py` | areas add up to the bounding box, and toggling all corners leaves exactly its 4 corners |
| Permutation Sequence | `math_geometry/permutation_sequence.py` | factorial number system: divmod the 0-based rank by (n−1)!, (n−2)!, ... and pop from the unused digits |
| Poor Pigs | `math_geometry/poor_pigs.py` | a pig has T + 1 outcomes; the answer is the smallest p with (T + 1)^p ≥ buckets |
| Random Pick Index | `practice/simple/basics/math/06_random_pick_and_reservoir_sampling.py` | reservoir of size 1 over the matches: keep the i-th match with probability 1/i |
| Reverse Bits | `bit_manipulation/reverse_bits.py` | peel x's lowest bit onto the right of the output exactly 32 times; leading zeros matter |
| Reverse Integer | `practice/simple/basics/math/04_digits_reverse_integer_and_palindrome_number.py` | peel the digits of `abs(x)` with divmod; check `rev > (LIMIT - d) // 10` before pushing a digit |
| Single Number | `bit_manipulation/single_number.py` | XOR everything: pairs cancel, the loner stays |
| Single Number III | `practice/simple/basics/bits/03_xor_tricks_single_number_missing_number.py` | XOR of all is a ^ b; its lowest set bit splits the numbers into two Single Number problems |

### Self-check

1. Why must the bit-array template test bit v *before* it sets it?
<details><summary>Answer</summary>Setting first makes the test always see a 1, so every value would look like a repeat. The test answers "had v appeared <em>before</em> this item?", so it has to run while the set still describes the earlier items only.</details>

2. `x & (x - 1)` and `x & -x`: which one deletes the lowest 1-bit and which one keeps only it?
<details><summary>Answer</summary><code>x - 1</code> flips the lowest 1 and every 0 below it, so <code>x &amp; (x - 1)</code> deletes exactly that 1. <code>-x</code> is <code>~x + 1</code>: it flips every bit <em>above</em> the lowest 1 and keeps the lowest 1 itself, so <code>x &amp; -x</code> keeps only that bit.</details>

3. Why does `while x: x >>= 1` hang for `x = -8`, and what are the two lines that fix it?
<details><summary>Answer</summary>Python's <code>&gt;&gt;</code> keeps the sign: -8 becomes -4, -2, -1, and <code>-1 &gt;&gt; 1</code> is -1 again, so x never reaches 0. Work on the 32-bit pattern: <code>x &amp;= 0xFFFFFFFF</code> first, then the same <code>while x: x &gt;&gt;= 1</code> ends after at most 32 shifts. Looping exactly 32 times works too.</details>

4. In Maximum XOR With an Element From Array, why sort the queries by their limit?
<details><summary>Answer</summary>Each query may only use numbers ≤ its limit. In increasing limit order the allowed sets only grow, so a single trie that we keep inserting into (never deleting from) holds exactly the allowed numbers when each query is answered. The answers are written back to the queries' original positions.</details>